# Entregável 1 — Especificação e Baseline

> **Grupo:** Rodolfo Dalla Costa, Thais Caroline Murer, Werner Conrado Jacob Denzin<br>
> **Tema/Projeto:** Sistema multiagente para suporte à perguntas (em linguagem natural) sobre dados educacionais brasileiros<br>
> **Disciplina:** INF0093 — Projeto Prático com Sistemas Multiagentes — 2S/2026<br>
> **Data:** 07/09/2026<br>

* Este notebook deve conter a **especificação inicial do sistema**, a implementação de um
**baseline funcional**, casos de teste e uma análise crítica das limitações observadas.
* Use células **Markdown** para documentação, justificativas e análise.
Use células **Python** para configuração, implementação, experimentos, testes e coleta de resultados.
* **Antes de enviar:** execute o notebook do início ao fim (`Runtime → Run all`) e salve com as
saídas visíveis. Um notebook sem saídas não permite avaliar o baseline.


# 1. Descrição do problema

O Brasil publica microdados educacionais e socioeconômicos de altíssima qualidade, sob licença aberta — como ENEM (INEP/MEC) e os agregados municipais do IBGE. Porém, na prática, para interpretá-los de modo efetivo, contribuindo, principalmente, para tomadas de decisão, há a necessidade de conhecimento técnico especializado. Por exemplo: os microdados do ENEM 2023 correspondem a um CSV de 1,8 GB com 76 colunas codificadas e o cruzamento com os dados do IBGE exige conhecimento adicional sobre APIs específicas que viabilizem tal agregação/correlação.<br>

Uma pergunta, por exemplo, do secretário municipal de educação — *"a média de matemática da minha cidade está acima ou abaixo da média do estado?"* — exigiria hoje horas de trabalho:<br>
* _(a) download dos dados;_<br>
* _(b) compreender o dicionário de variáveis;_<br>
* _(c) selecionar as variáveis relevantes para esta pergunta;_<br>
* _(d) escrever a `query/groupby`;_<br>
* _(e) achar o código IBGE do município; e_ <br>
* _(f) interpretar o resultado_
<br><br>

| Característica | Descrição |
|---|---|
| **Relevância** | Quem toma decisão sobre educação municipal — secretarias, conselhos, imprensa local, ONGs — raramente possui uma equipe de Ciência de Dados, cenário que influencia diretamente a qualidade e a viabilidade de análises específicas e customizadas sobre um determinado domínio, levando esses atores a recorrer a estudos e rankings de terceiros publicados anualmente. |
| **Contexto** | Uso analítico e exploratório para a realização de perguntas concretas e específicas sobre um município, um estado ou uma região, com a garantia de obter resultados acurados, seguros para tomadas de decisão e planejamento educacional. |
| **Objetivo** | Prover uma interface através da qual usuários possam realizar perguntas em linguagem natural (Português) sobre a qualidade e o desempenho educacional (e socioeconômico) a partir de fontes de dados confiáveis, como ENEM e IBGE — um sistema totalmente transparente, que abstrai qualquer necessidade de conhecimento sobre as bases de dados ou técnico-analítico. |
| **Escopo** | Esta **1a versão** tem por objetivo: *(a) receber uma pergunta do usuário; (b) interpretá-la; (c) consultar os dados necessários para respondê-la (sobre bases do ENEM 2023 e IBGE); e (d) gerar a resposta*. Fora do escopo — gráficos, validação dos resultados, memória e arquitetura multiagente — aprimoramentos previstos para as próximas versões (roadmap). |


# 2. Usuário-alvo e stakeholders

| Usuário | Categoria | Objetivos |
|---|---|---|
| Analista educacional (secretaria municipal/estadual, jornalismo de dados, terceiro setor) | Principal | Obter, sob prazo, um número confiável sobre um município ou recorte geográfico, sem depender de um programador |
| Pesquisador em educação | Secundário | Definir hipóteses, verificar, validar resultados/dados, iterar com novas hipóteses |
| Gestor escolar / conselho municipal | Secundário | Comparar o município com vizinhos e com a média estadual, não apenas obter valores isolados |
| Jornalista de educação | Secundário | Verificar uma afirmação de fonte oficial, conhecendo o recorte exato para evitar publicações incorretas |
| INEP/MEC e IBGE | Stakeholder | Garantir que o dado seja usado sem ser deturpado; toda resposta do sistema precisa declarar o recorte aplicado |
| Secretarias de educação | Stakeholder | Consumir números confiáveis para publicação — um número errado tem custo político real |
| Sociedade / imprensa | Stakeholder | Beneficiar-se, indiretamente, de um dado público mais acessível |

# 3. Casos de uso principais
> Para esta 1a versão foram selecionados os seguintes casos de uso (UC)

| # | UC | Ator | Entrada | Objetivo | Saída esperada | Sucesso |
|---|--|---|---|---|---|---|
| 1 | Consulta simples | Analista (secretaria municipal) | Qual é a média geral do ENEM 2023 em Campinas? | Obter um valor único sobre um município | Resposta em Português com a média solicitada | O valor confere com a consulta de referência |
| 2 | Ranking com filtro | Jornalista de educação | Quais são os 5 municípios de SP com maior média em matemática entre os que têm ao menos 100 participantes? | Obter lista ordenada, com os filtros aplicados | Nomes dos municípios que satisfaçam as condições desejadas | Os valores conferem com a consulta de referência |
| 3 | Cruzamento ENEM × IBGE | Pesquisador em educação | Qual é a correlação entre o PIB per capita e a média geral dos municípios com pelo menos 50 participantes? | Relacionar desempenho educacional e indicador socioeconômico | Coeficiente de correlação e a ressalva de que ENEM é de 2023 e IBGE de 2021 | O coeficiente confere com a consulta de referência e a resposta menciona a diferença de anos |
| 4 | Fora do domínio da solução | Qualquer usuário | Qual é o IDH de Campinas? | Coerência e evitar alucinação | Resposta dizendo que a pergunta está fora do domínio da solução | A solução se mantém íntegra, dizendo que não tem informações para responder |

# 4. Escopo, não-objetivos e premissas

### 4.1 Escopo

| Item | Descrição |
|---|---|
| **Dados** | ENEM 2023 agregado por município da escola × população e PIB municipais do IBGE (2021) |
| **Interação** | Uma pergunta por vez, em português, sem histórico (*stateless*) |
| **Saída** | Texto em português + valor numérico/tabela + código pandas executado |
| **Arquitetura** | Uma chamada ao LLM que planeja a consulta; execução e formatação da resposta |

### 4.2 Não-objetivos

| Fora do escopo | Por quê |
|---|---|
| Geração de gráficos | é o agente *Visualizador*, previsto dentre os próximos entregáveis; adicioná-lo nesta versão, não permitirá avaliar seu ganho efetivo para a solução |
| Validação automática do resultado com *retry* | é o agente *Validador*; o baseline **precisa** errar de forma observável para que o ganho seja mensurável |
| Arquitetura multiagente (LangGraph) | apesar de um requisito, esta versão deve ser a mais simples - pois, adição de complexidade sem necessidade deve ser evitada |
| Memória de sessão | *stateless* é mais simples de avaliar; memória será adicionada dentre os próximos entregáveis do projeto |
| Microdados individuais (nota por candidato, questionário socioeconômico) | Volume de 1,8 GB excede dinâmica (viabilidade) para fluxo interativo |
| Séries temporais (ENEM de vários anos) | Escopo reduzido para validação do sistema para apenas 1 ano (2023) |
| Interface web / API pública | Jupyter notebook é a única interface disponível para validação do sistema |

### 4.3 Premissas

* Para não onerar o processamento do notebook, a seção 12 (Dados) descreve a rotina de ETL/pré-processamento desenvolvida para gerar o arquivo utilizado pelo projeto, `enem2023_ibge_municipios.csv`
* O arquivo `enem2023_ibge_municipios.csv` é viável de ser carregado em memória como seu esquema na janela de contexto dos LLMs
* O código de município do INEP (`CO_MUNICIPIO_ESC`) corresponde 1:1 ao código do IBGE — **verificado no ETL**: 5.481 de 5.481 municípios
* O serviço de inferência (Groq) está disponível e `temperature=0` reduz — mas não elimina — a variação entre execuções

# 5. Entradas e saídas

### 5.1 Entradas

Uma **string em português**: pergunta em linguagem natural sobre o recorte ENEM 2023 ×
IBGE. Exemplos representativos:

- `"Qual é a média geral do ENEM 2023 em Campinas?"` — valor único
- `"Quantos municípios têm média geral acima de 550?"` — contagem com filtro
- `"Qual região tem a maior média de redação?"` — agregação por grupo
- `"Qual é a correlação entre PIB per capita e média geral?"` — cruzamento ENEM × IBGE
- `"Qual é a nota média de inglês em Salvador?"` — **fora do recorte**, deve gerar abstenção

### 5.2 Saídas

Um objeto estruturado (validado por Pydantic) com:

| Campo | Tipo | Conteúdo |
|---|---|---|
| `texto` | str | Resposta em português, com o valor já formatado, seguida da nota de recorte |
| `resultado` | escalar / Series / DataFrame | Valor bruto devolvido pela execução do código |
| `codigo` | str | Expressão pandas efetivamente executada (evidência auditável) |
| `viavel` | bool | `False` quando a pergunta não é respondível com as colunas disponíveis |
| `motivo` | str | Estratégia adotada, ou o que falta nos dados quando `viavel=False` |
| `erro_execucao` | str \| None | Mensagem de erro, quando o código gerado falha |

* Acompanham a saída as **métricas de execução** (latência, tokens de entrada e saída,
número de chamadas ao LLM), exigidas pela seção 3.2 do enunciado.

#### 5.2.1 Exemplo:

```python
Resposta(
    texto=(
        "A média geral do ENEM 2023 em Campinas, SP, é 584.24.\n\n"
        "Recorte: ENEM 2023, apenas candidatos com escola declarada, presentes nos "
        "dois dias e com as cinco notas (721.429 de ~3,9 milhões de inscritos), "
        "agregados pelo município da escola. Indicadores do IBGE são de 2021."
    ),
    resultado=584.24,
    codigo='df.loc[(df.municipio == "Campinas") & (df.uf == "SP"), "media_geral"].item()',
    viavel=True,
    motivo="",
    erro_execucao=None
)
```

# 6. Requisitos funcionais

Cada requisito abaixo passa no teste *"consigo escrever hoje o critério que decide se ele
foi atendido?"*. A coluna **Como será verificado** aponta para a seção 15 (implementação
da verificação) e para os casos da seção 14.

| ID | Requisito | Como será verificado |
|---|---|---|
| **RF-01** | Responder perguntas de **valor único** (média, contagem, máximo) sobre o recorte: em 11 casos automáticos, acertar ao menos 8. | Comparação numérica do resultado executado com a resposta de referência, tolerância relativa de 1% (`resultado_correto`, casos T01, T02, T05) |
| **RF-02** | Responder perguntas de **ranking**, respeitando filtros de robustez declarados na pergunta (ex.: `n_participantes >= 100`). | Os nomes de referência aparecem na resposta final; o código gerado contém o filtro (`cobertura_lista`, caso T03) |
| **RF-03** | Responder perguntas que **cruzam ENEM e IBGE** (correlação, comparação entre grupos, recorte por população). | Comparação numérica com a referência (casos T06, T07, T08) |
| **RF-04** | Não produzir número que não venha da execução sobre o `DataFrame`: nenhum valor pode vir do conhecimento do modelo. | Toda resposta aprovada precisa ter `codigo` não vazio e execução sem erro; números no texto que não aparecem no resultado reprovam (`texto_fiel`) |
| **RF-05** | **Abster-se** quando a pergunta exigir informação ausente do recorte (coluna inexistente, outro ano, outro exame), nomeando o que falta. | `viavel == False`, nenhum código executado e presença de marcador de ausência no texto (`abstencao_valida`, casos T09–T11) |
| **RF-06** | Expor como **evidência** o código pandas executado e o resultado bruto. | Presença não vazia de `codigo` e `resultado` em toda resposta viável (verificação estrutural na seção 16) |
| **RF-07** | Declarar as **limitações do recorte** (subconjunto de candidatos, anos diferentes entre ENEM e IBGE) junto da resposta. | O texto final contém a nota de recorte; ver limitação registrada na seção 18 — no baseline a nota é **fixa**, não sensível à pergunta |
| **RF-08** | Gerar **gráfico** quando o usuário pedir. | **Não atendido nesta versão** (não-objetivo declarado na seção 4); entra no Entregável 3 |
| **RF-09** | Pedir esclarecimento diante de pergunta **ambígua** (ex.: "média de São Paulo" — capital ou estado?). | Rubrica manual, casos T12 e T13; sem verificação automática nesta versão |

* RF-08 e RF-09 estão escritos aqui **de propósito**, mesmo sem serem atendidos: são a "régua" que tornará mensurável o ganho das próximas arquiteturas

# 7. Requisitos não funcionais e restrições

| ID | Requisito | Valor de referência | Como será verificado |
|---|---|---|---|
| **RNF-01** | Saída estruturada e validada | esquema Pydantic; 0 erros de *parsing* nos 13 casos | campo `erro_parse` registrado por caso na seção 16 |
| **RNF-02** | Latência por consulta | mediana **< 10 s** | `latencia_s` medida por caso; mediana na seção 17 |
| **RNF-03** | Custo de chamadas ao LLM | **≤ 1 chamada** por pergunta no baseline; custo total do conjunto **< US$ 0,05** | `chamadas_llm` e contagem de tokens (`usage_metadata`) × preço da Groq |
| **RNF-04** | Reprodutibilidade | `temperature=0`; modelo, versão do prompt, data e versão do Python registrados em `RUN_INFO` e salvos junto dos resultados | arquivo `baseline_v1_resultados.json` |
| **RNF-05** | Rastreabilidade | para todo caso é possível reconstruir pergunta → código → resultado bruto → texto final | colunas `codigo`, `resultado_bruto` e `resposta` na tabela da seção 17 |
| **RNF-06** | Segurança da execução | o código gerado não pode importar módulos, acessar arquivos, rede ou atributos privados | lista de tokens proibidos aplicada **antes** de executar (`codigo_seguro`, seção 13) |
| **RNF-07** | Custo de preparação dos dados | o sistema **não** processa 1,8 GB em tempo de consulta; carrega um artefato de 0,6 MB | tempo de carga medido na seção 12 |

### 7.1 Restrições

- **Modelo fixo pela disciplina** (`openai/gpt-oss-20b` ou `llama-3.3-70b-versatile` via Groq);
  não há *fine-tuning*.
- **Nenhuma chave de API escrita no notebook** — carregada de `userdata` (Colab) ou de
  variável de ambiente.
- `temperature=0` reduz variação, mas **não garante** saídas idênticas em serviço de
  inferência distribuída; conclusões sobre diferença entre versões exigem repetição.
- O artefato de dados é congelado junto com o conjunto de avaliação: mudar o ETL obriga a
  reexecutar o baseline.


# 8. Recursos externos potencialmente necessários

### 8.1 Já em uso nesta versão

| Recurso | Papel | Situação |
|---|---|---|
| Groq API (`openai/gpt-oss-20b`) | planejamento da consulta (1 chamada) | disponível |
| `langchain` / `langchain-groq` | cliente e saída estruturada (`with_structured_output`) | disponível |
| `pydantic` | validação do esquema de saída (RNF-01) | disponível |
| `pandas` | execução da consulta sobre o artefato | disponível |
| INEP — Microdados do ENEM 2023 | fonte primária das notas | baixado uma vez no ETL (seção 12) |
| IBGE — API de Agregados v3 (6579 e 5938) | população estimada e PIB municipal de 2021 | baixado uma vez no ETL |
| IBGE — API de Localidades v1 | nome, UF e região de cada município | baixado uma vez no ETL |

### 8.2 Hipóteses para as próximas versões

| Recurso | Para quê |
|---|---|
| `langgraph` | grafo Planejador → Supervisor → agentes especializados |
| Log estruturado em JSON (um registro por transição de agente) | observabilidade; base para discutir confiabilidade |
| `matplotlib` | agente Visualizador (RF-08) |
| LLM como juiz, calibrado contra os casos manuais | avaliar respostas abertas e ambíguas (RF-09) |
| Servidor MCP para a API do IBGE | consultar indicadores fora do artefato pré-processado, sem novo ETL |
| Microdados do ENEM de outros anos | séries temporais |

* Nenhum destes será adotado agora, mas justificado, no entregável em que entrar, por alguma limitação **observada** na seção 18

# 9. Tipo de baseline escolhido

* Classificação: **parcial**

  * O baseline executa o núcleo da tarefa — pergunta em português → consulta correta sobre o
recorte → resposta em português com evidência — para perguntas de **valor único, ranking e
cruzamento ENEM × IBGE**, que são os casos de uso UC-01 a UC-04. 
  * Ele **não** executa a
tarefa completa descrita na proposta: não gera gráfico (RF-08), não valida o próprio
resultado, não pede esclarecimento diante de ambiguidade (RF-09) e não adapta as ressalvas
à pergunta (RF-07 é atendido por uma nota fixa).

* Por que essa escolha é adequada?

  * É a solução mais simples que ainda resolve o problema declarado: **uma chamada ao LLM**
para planejar a consulta, seguida de execução e formatação **determinísticas**. 
  * Uma baseline mais simples — LLM respondendo direto, sem tocar nos dados — seria um espantalho: ele
alucinaria números com uma baixa taxa de acerto, tornando qualquer arquitetura posterior
"melhor" sem informação. 
  * Um baseline mais complexo — já com validador e *retry* — gastaria
agora o incremento do Entregável 2 e apagaria justamente o ganho que se quer medir.

* O que foi deliberadamente simplificado ou excluído?

| Simplificação | Consequência esperada |
|---|---|
| Nenhuma validação do resultado | erros de coluna, de filtro ou de agregação passam direto para o usuário |
| Nenhum *retry* | um código que falha ao executar produz resposta vazia, não uma segunda tentativa |
| Uma única chamada ao LLM | o modelo escreve o texto **antes** de ver o resultado; a frase final é um *template* com um marcador `{resultado}` |
| Nota de limitação fixa | a ressalva sobre anos diferentes aparece mesmo quando a pergunta não cruza ENEM e IBGE |
| Sem memória | *"e no ano anterior?"* não tem como funcionar |

* Como permitirá a comparação futura?

  * O conjunto de avaliação (seção 14) é **congelado** e os resultados são salvos em
`baseline_v1_resultados.json`. 
  * Nos Entregáveis 2, 3 e 4 os mesmos 13 casos
serão executados contra a arquitetura da vez e comparados nas mesmas cinco medidas:
taxa de aprovação, acerto por tipo de caso, latência mediana, chamadas ao LLM e custo.
  * A pergunta que a comparação responde é direta: **cada agente acrescentado paga o custo de
latência e de tokens que ele impõe?**


# 10. Critérios preliminares de sucesso

| Critério | Requisito | Como será medido | Meta nesta versão |
|---|---|---|---|
| **Correção numérica** | RF-01, RF-03 | verificação determinística: valor executado × resposta de referência, tolerância relativa de 1% | ≥ 8 de 11 casos automáticos |
| **Cobertura de lista** | RF-02 | fração dos itens de referência presentes na resposta final | 100% no caso de ranking (T03) |
| **Fidelidade ao resultado** | RF-04 | o valor mostrado no texto é o valor devolvido pela execução | 100% das respostas viáveis |
| **Abstenção correta** | RF-05 | `viavel=False`, sem código executado, com marcador de ausência no texto | 3 de 3 (T09, T10, T11) |
| **Ausência de falsa abstenção** | RF-05 | o sistema **não** se abstém em pergunta respondível | 0 falsas abstenções nos casos T01–T08 |
| **Evidência disponível** | RF-06 | `codigo` e `resultado` não vazios em toda resposta viável | 100% |
| **Erro de execução** | RNF-01, RNF-06 | fração de casos em que o código gerado levanta exceção ou é bloqueado | ≤ 1 de 13 |
| **Latência** | RNF-02 | mediana de `latencia_s` sobre os 13 casos | < 10 s |
| **Custo** | RNF-03 | (tokens de entrada × preço + tokens de saída × preço); preços em <https://groq.com/pricing> | < US$ 0,05 no conjunto |
| **Ambiguidade** | RF-09 | **rubrica manual**: a resposta distingue as leituras possíveis ou declara a que adotou? Notas registradas na seção 17 | qualitativo, sem meta |

### 10.1 Sobre a escolha das formas de verificação

- **Verificação determinística** para tudo que é numérico ou lista fechada — mais barata,
  e objetiva
- **Rubrica humana** apenas para os dois casos ambíguos (T12, T13), onde não existe
  resposta única certa; as notas ficam registradas no notebook.
- **LLM como juiz** não é usado nesta versão: com 11 casos automáticos verificáveis por
  comparação numérica, ele acrescentaria custo e uma fonte de erro sem necessidade.

# 11. Configuração do ambiente

* A célula de chave procura `GROQ_API_KEY`, nesta ordem: variável de ambiente → arquivo
  `.secret` no diretório atual ou em até quatro níveis acima (fica em `eval/.secret`,
  compartilhado por todos os entregáveis; ignorado pelo git; modelo em `eval/.secret.example`)
  → `userdata` do Colab (nome `INF0093-2026-2S`, o mesmo do material da disciplina) → teclado.
* **Nenhuma chave é escrita no notebook** (exigência do enunciado, seção 4).
* `RUN_INFO` registra modelo, temperatura, versão do prompt e data: sem esse registro os
  números desta execução não podem ser comparados com os das próximas arquiteturas
  (seção 3.2 do enunciado).


In [1]:
%pip install -q -U langchain langchain-groq pydantic pandas


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import ast
import datetime
import getpass
import io
import json
import os
import platform
import re
import ssl
import time
import unicodedata
import urllib.request
import zipfile
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

In [3]:
def localizar_secret(inicio: Path = Path.cwd(), niveis: int = 4) -> Path | None:
    """Procura `.secret` no diretório atual e nos `niveis` diretórios acima.
    """
    for pasta in [inicio, *inicio.parents[:niveis]]:
        if (pasta / ".secret").is_file():
            return pasta / ".secret"
    return None

def carregar_chave_groq() -> str:
    """Procura a chave, nesta ordem: variável de ambiente, arquivo .secret,
    userdata do Colab, teclado. Nunca a exibe."""
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"

    # Aceita `GROQ_API_KEY=gsk_...` ou só a chave. Modelo em eval/.secret.example.
    arquivo = localizar_secret()
    if arquivo:
        for linha in arquivo.read_text(encoding="utf-8").splitlines():
            linha = linha.strip()
            if not linha or linha.startswith("#"):
                continue
            chave = (linha.split("=", 1)[1] if "=" in linha else linha).strip().strip('"').strip("'")
            if chave:                                # placeholder vazio: segue adiante
                os.environ["GROQ_API_KEY"] = chave
                return f"arquivo {arquivo.parent.name}/.secret"

origem = carregar_chave_groq()
assert os.environ.get("GROQ_API_KEY"), "Chave não configurada."
print("Chave carregada via:", origem)

Chave carregada via: arquivo eval/.secret


In [4]:
from langchain_groq import ChatGroq

# MODEL_NAME = "llama-3.3-70b-versatile"   # Meta
MODEL_NAME = "openai/gpt-oss-20b"          # OpenAI

TEMPERATURE = 0
PROMPT_VERSAO = "v1"

llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

# Registro da execução
RUN_INFO = {
    "modelo": MODEL_NAME,
    "temperatura": TEMPERATURE,
    "prompt_versao": PROMPT_VERSAO,
    "data": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
}
RUN_INFO

{'modelo': 'openai/gpt-oss-20b',
 'temperatura': 0,
 'prompt_versao': 'v1',
 'data': '2026-09-07T12:38:58',
 'python': '3.13.12'}

# 12. Dados

### 12.1 Informações

**Dataset do projeto**

* O sistema **não lê os microdados do ENEM durante a consulta**.

* O dataset utilizado pelo sistema é produzido previamente por um ETL executado fora do sistema. Esse processo baixa os dados do INEP, que ocupam cerca de 550 MB, lê o CSV de aproximadamente 1,8 GB em blocos, agrega os resultados por município da escola e realiza o cruzamento com duas APIs do IBGE.

* O resultado desse processamento é um CSV de aproximadamente **0,6 MB**, com 5.481 linhas. Esse tamanho permite manter o arquivo em memória e descrever todo o seu esquema no prompt (RNF-07).

* O código responsável pelo ETL está disponível na seção 12.5. Ele está definido no notebook, mas não é executado durante o fluxo normal, evitando que o processamento seja repetido a cada `Run all`.

**Filtros e critérios de seleção**

Para entrar na agregação, o participante precisa atender simultaneamente aos três critérios abaixo:

1. ter declarado o município da escola (`CO_MUNICIPIO_ESC` preenchido);
2. ter participado dos dois dias de prova; e
3. possuir as cinco notas disponíveis.

Com esses filtros, o dataset final reúne **721.429 participantes**, de aproximadamente 3,9 milhões de inscritos no ENEM 2023.

**Limitações dos dados**

1. **Recorte dos candidatos.** O subconjunto utilizado é formado principalmente por concluintes do ensino médio regular que declararam uma escola. Portanto, as médias calculadas **não representam a média de todos os estudantes do município**.

2. **Município da escola ≠ município de residência ≠ município de prova.** O município utilizado na agregação é o da escola. Essa escolha foi feita de forma deliberada por ser a referência com maior relação com o contexto educacional analisado, mas ela pode produzir resultados diferentes daqueles obtidos usando residência ou local de prova.

3. **Anos diferentes.** Os dados do ENEM são de 2023, enquanto os dados socioeconômicos do IBGE utilizados no cruzamento são de 2021. Por isso, as comparações entre desempenho e indicadores socioeconômicos devem ser interpretadas como aproximações, e não como uma fotografia do mesmo período.

4. **Cobertura do cruzamento.** Os 5.481 códigos de município presentes nos dados do INEP tiveram correspondência 1:1 com os códigos do IBGE. Portanto, o problema de compatibilidade entre códigos de município identificado como risco na proposta do projeto **não ocorreu nesta versão do dataset**.


### 12.2 Dicionário dos dados — `enem2023_ibge_municipios.csv`

| Coluna | Tipo | Descrição |
|---|---|---|
| `co_municipio` | int | Código IBGE do município (7 dígitos). |
| `municipio` | str | Nome do município. |
| `uf` | str | Sigla da unidade da federação (`SP`, `BA`, ...). |
| `regiao` | str | Norte, Nordeste, Centro-Oeste, Sudeste, Sul. |
| `populacao_2021` | float | População residente estimada em 2021 (IBGE, agregado 6579, variável 9324). |
| `pib_2021_mil_reais` | float | PIB municipal a preços correntes de 2021, em mil reais (IBGE, agregado 5938, variável 37). |
| `pib_per_capita_2021` | float | `pib_2021_mil_reais * 1000 / populacao_2021`, em reais por habitante. Calculado no ETL. |
| `n_participantes` | int | Candidatos do município considerados na média (ver critério de inclusão acima). |
| `media_cn` | float | Média da nota de Ciências da Natureza. |
| `media_ch` | float | Média da nota de Ciências Humanas. |
| `media_lc` | float | Média da nota de Linguagens e Códigos. |
| `media_mt` | float | Média da nota de Matemática. |
| `media_redacao` | float | Média da nota de Redação. |
| `media_geral` | float | Média das cinco notas por candidato, agregada por município. |
| `pct_escola_publica` | float | Percentual dos participantes cuja escola é da rede pública (`TP_ESCOLA == 2`). |

### 12.3 Fontes

- INEP/MEC, Microdados do ENEM 2023 — <https://download.inep.gov.br/microdados/microdados_enem_2023.zip>
- IBGE, API de Agregados v3, agregado 6579 (população estimada 2021)
- IBGE, API de Agregados v3, agregado 5938 (PIB municipal 2021)
- IBGE, API de Localidades v1 (nome, UF e região dos municípios)

### 12.4 Load  `enem2023_ibge_municipios.csv`

* O CSV final está publicado junto com o notebook, no repositório do projeto no GitHub. 
* A célula abaixo baixa esse artefato diretamente da URL pública.

In [5]:
URL_DADOS = ("https://raw.githubusercontent.com/werner-denzin/unicamp.agents/main/"
             "04_projeto_sistema_multiagentes/eval/data/"
             "enem2023_ibge_municipios.csv")

inicio = time.perf_counter()
df = pd.read_csv(URL_DADOS)
origem_dados = URL_DADOS
TEMPO_CARGA = time.perf_counter() - inicio

print(f"origem   : {origem_dados}")
print(f"formato  : {df.shape[0]} municípios × {df.shape[1]} colunas")
print(f"carga    : {TEMPO_CARGA:.2f} s   (RNF-07)")
print(f"nulos    : {int(df.isna().sum().sum())}")
print(f"cobertura: {int(df['n_participantes'].sum()):,} participantes".replace(",", "."))
df.head(3)

origem   : https://raw.githubusercontent.com/werner-denzin/unicamp.agents/main/04_projeto_sistema_multiagentes/eval/data/enem2023_ibge_municipios.csv
formato  : 5481 municípios × 15 colunas
carga    : 0.32 s   (RNF-07)
nulos    : 0
cobertura: 721.429 participantes


,co_municipio,municipio,uf,regiao,populacao_2021,pib_2021_mil_reais,n_participantes,media_cn,media_ch,media_lc,media_mt,media_redacao,media_geral,pct_escola_publica,pib_per_capita_2021
0,1200013,Acrelândia,AC,Norte,15721.0,398725.0,34,476.39,486.03,493.29,486.84,560.00,500.51,100.0,25362.57
1,1200054,Assis Brasil,AC,Norte,7649.0,133916.0,8,462.34,500.61,494.55,455.30,437.50,470.06,100.0,17507.65
2,1200104,Brasiléia,AC,Norte,27123.0,685636.0,65,458.61,484.94,486.28,469.83,531.38,486.21,100.0,25278.77


### 12.5 ETL / Código

* O código abaixo contém o pipeline completo utilizado para gerar o CSV carregado na célula anterior.

* O processo baixa os microdados brutos do INEP e as fontes de dados do IBGE, processa os dados do ENEM em blocos para realizar a agregação por município e, por fim, cruza os dois conjuntos. Todo o processamento é feito em memória, sem a criação de arquivos intermediários no disco.

* As funções são apenas definidas na célula, portanto sua execução leva poucos milissegundos. O pipeline, no entanto, **não é chamado automaticamente**. Reprocessar 1,8 GB de dados a cada `Run all` seria desnecessário e contrariaria o objetivo do RNF-07 (seção 7).

* A chamada do pipeline permanece comentada no final da célula. Quando executada manualmente, ela reproduz o `DataFrame` **em memória**, armazenado na variável `df_reproduzido`, sem gerar nenhum arquivo.

* Há ainda um segundo trecho, também comentado, que mostra como salvar esse `DataFrame` em `eval/data/enem2023_ibge_municipios.csv`, caso seja necessário regenerar o arquivo utilizado pelo projeto.


In [6]:
URL_ENEM = "https://download.inep.gov.br/microdados/microdados_enem_2023.zip"
URL_POP = ("https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2021"
           "/variaveis/9324?localidades=N6[all]")
URL_PIB = ("https://servicodados.ibge.gov.br/api/v3/agregados/5938/periodos/2021"
           "/variaveis/37?localidades=N6[all]")
URL_MUN = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
SUFIXO_CSV = "DADOS/MICRODADOS_ENEM_2023.csv"

COLUNAS_ENEM = [
    "CO_MUNICIPIO_ESC", "NO_MUNICIPIO_ESC", "SG_UF_ESC", "TP_ESCOLA",
    "TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT",
    "NU_NOTA_CN", "NU_NOTA_CH", "NU_NOTA_LC", "NU_NOTA_MT", "NU_NOTA_REDACAO",
]
AREAS = ["CN", "CH", "LC", "MT"]


def baixar_bytes(url: str, verificar_tls: bool = True) -> bytes:
    """Baixa `url` inteiramente em memória — nenhum arquivo é gravado em disco."""
    contexto = None if verificar_tls else ssl._create_unverified_context()
    with urllib.request.urlopen(url, context=contexto) as r:
        return r.read()


def ler_agregado(bruto: bytes, coluna: str) -> pd.DataFrame:
    """Achata a resposta da API de Agregados v3 em (co_municipio, <coluna>)."""
    linhas = []
    for variavel in json.loads(bruto.decode("utf-8")):
        for resultado in variavel["resultados"]:
            for serie in resultado["series"]:
                (_, valor), = serie["serie"].items()
                linhas.append({
                    "co_municipio": int(serie["localidade"]["id"]),
                    coluna: pd.to_numeric(valor, errors="coerce"),
                })
    return pd.DataFrame(linhas)


def ler_municipios(bruto: bytes) -> pd.DataFrame:
    """Nome, UF e região de cada município (municípios novos só têm UF em regiao-imediata)."""
    def uf_de(m: dict) -> dict:
        micro = m.get("microrregiao")
        if micro:
            return micro["mesorregiao"]["UF"]
        return m["regiao-imediata"]["regiao-intermediaria"]["UF"]

    dados = json.loads(bruto.decode("utf-8"))
    return pd.DataFrame([{
        "co_municipio": m["id"],
        "municipio": m["nome"],
        "uf": uf_de(m)["sigla"],
        "regiao": uf_de(m)["regiao"]["nome"],
    } for m in dados])


def agregar_enem(zip_bruto: bytes, tamanho_bloco: int = 400_000) -> pd.DataFrame:
    """Percorre o CSV de 1,8 GB em blocos e soma notas por município da escola.

    Participante = presente nos dois dias (as quatro áreas) e com as cinco notas
    preenchidas; só entram candidatos com escola declarada, vínculo municipal
    usado no cruzamento com o IBGE.
    """
    parciais = []
    notas = [f"NU_NOTA_{a}" for a in AREAS] + ["NU_NOTA_REDACAO"]

    with zipfile.ZipFile(io.BytesIO(zip_bruto)) as z:
        interno = next(n for n in z.namelist() if n.endswith(SUFIXO_CSV))
        with z.open(interno) as fluxo:
            leitor = pd.read_csv(
                fluxo, sep=";", encoding="latin-1", usecols=COLUNAS_ENEM,
                chunksize=tamanho_bloco, low_memory=False,
            )
            for bloco in leitor:
                presente = (bloco[[f"TP_PRESENCA_{a}" for a in AREAS]] == 1).all(axis=1)
                bloco = bloco[bloco["CO_MUNICIPIO_ESC"].notna()
                              & presente
                              & bloco[notas].notna().all(axis=1)]
                if bloco.empty:
                    continue
                bloco = bloco.assign(
                    co_municipio=bloco["CO_MUNICIPIO_ESC"].astype("int64"),
                    media_geral=bloco[notas].mean(axis=1),
                    publica=(bloco["TP_ESCOLA"] == 2).astype("int64"),
                )
                parciais.append(bloco.groupby("co_municipio").agg(
                    n_participantes=("media_geral", "size"),
                    soma_cn=("NU_NOTA_CN", "sum"),
                    soma_ch=("NU_NOTA_CH", "sum"),
                    soma_lc=("NU_NOTA_LC", "sum"),
                    soma_mt=("NU_NOTA_MT", "sum"),
                    soma_red=("NU_NOTA_REDACAO", "sum"),
                    soma_geral=("media_geral", "sum"),
                    n_publica=("publica", "sum"),
                ))

    total = pd.concat(parciais).groupby(level=0).sum()
    saida = pd.DataFrame({
        "n_participantes": total["n_participantes"],
        "media_cn": total["soma_cn"] / total["n_participantes"],
        "media_ch": total["soma_ch"] / total["n_participantes"],
        "media_lc": total["soma_lc"] / total["n_participantes"],
        "media_mt": total["soma_mt"] / total["n_participantes"],
        "media_redacao": total["soma_red"] / total["n_participantes"],
        "media_geral": total["soma_geral"] / total["n_participantes"],
        "pct_escola_publica": 100 * total["n_publica"] / total["n_participantes"],
    }).reset_index()
    return saida


def construir_dataset() -> pd.DataFrame:
    """Reproduz, do zero, o CSV carregado na célula anterior a partir das fontes brutas."""
    enem = agregar_enem(baixar_bytes(URL_ENEM, verificar_tls=False))
    df_bruto = (ler_municipios(baixar_bytes(URL_MUN))
                .merge(ler_agregado(baixar_bytes(URL_POP), "populacao_2021"),
                       on="co_municipio", how="left")
                .merge(ler_agregado(baixar_bytes(URL_PIB), "pib_2021_mil_reais"),
                       on="co_municipio", how="left")
                .merge(enem, on="co_municipio", how="inner"))

    df_bruto["pib_per_capita_2021"] = (
        df_bruto["pib_2021_mil_reais"] * 1_000 / df_bruto["populacao_2021"]
    ).round(2)
    numericas = [c for c in df_bruto.columns if c.startswith(("media_", "pct_"))]
    df_bruto[numericas] = df_bruto[numericas].round(2)
    return df_bruto.sort_values(["uf", "municipio"]).reset_index(drop=True)


# Descomentar para construir o CSV do zero a partir das fontes brutas (leva vários minutos):
# df_reproduzido = construir_dataset()
# Salva df construido do zero
# df_reproduzido.to_csv("../../data/enem2023_ibge_municipios.csv", index=False, encoding="utf-8") 

### 12.6 ETL / Overview

* **Fontes.** O pipeline utiliza os microdados brutos do ENEM 2023, disponibilizados pelo INEP em um arquivo compactado de aproximadamente 550 MB, além de três fontes do IBGE: população estimada de 2021 (agregado 6579), PIB municipal de 2021 (agregado 5938) e a malha de municípios, utilizada para obter nome, UF e região.

* **Processamento em memória.** A função `baixar_bytes` carrega cada fonte diretamente em memória, como `bytes`. Dessa forma, nenhum arquivo intermediário é gravado em disco durante o processamento, incluindo o ZIP do INEP e os arquivos JSON retornados pelas APIs do IBGE.

* **TLS do INEP.** O certificado de `download.inep.gov.br` não é validado corretamente pela cadeia de certificados padrão. Por isso, a verificação TLS é desativada exclusivamente para essa fonte, por meio do parâmetro `verificar_tls=False`.

* **Agregação do ENEM (`agregar_enem`).** O CSV de aproximadamente 1,8 GB é processado em blocos de 400 mil linhas, evitando que todo o arquivo precise ser carregado na memória de uma só vez. As notas são agregadas por `CO_MUNICIPIO_ESC` e, antes da agregação, são mantidos apenas os participantes que declararam uma escola, estiveram presentes nos dois dias de prova (quatro áreas) e possuem as cinco notas preenchidas.

* **Dados do IBGE (`ler_agregado`, `ler_municipios`).** As funções transformam os retornos JSON das APIs em estruturas mais simples. Os dados da API de Agregados v3 são convertidos em pares `(co_municipio, valor)`, enquanto os dados de Localidades são organizados como `(co_municipio, nome, uf, região)`. O código também trata o caso de municípios mais recentes, nos quais a informação da UF pode aparecer apenas em `regiao-imediata`.

* **Cruzamento (`construir_dataset`).** Os dados de municípios, população, PIB e resultados agregados do ENEM são combinados pelo código do município. O `inner join` é utilizado especificamente na junção com os dados do ENEM, evitando que municípios sem candidatos selecionados entrem no dataset final. Nessa etapa também é calculado o `pib_per_capita_2021`, as colunas de médias e percentuais são arredondadas e o resultado é ordenado por UF e nome do município.

* **Por que o pipeline não roda automaticamente.** A célula apenas define as funções, o que tem custo computacional desprezível. A chamada a `construir_dataset()` permanece comentada para evitar o reprocessamento dos 1,8 GB de dados a cada `Run all`. Executar o pipeline automaticamente dessa forma seria desnecessário e contrariaria o objetivo do RNF-07 (seção 7).


### 12.7 Esquema entregue ao modelo

* O modelo nunca vê os dados, vê apenas a descrição abaixo. 
* Ela é gerada a partir do próprio `DataFrame`, para não sair de sincronia com o arquivo, e é o único contexto que o baseline tem para decidir **se** a pergunta é respondível e **como** respondê-la.


In [7]:
DESCRICOES = {
    "co_municipio":        "código IBGE do município (7 dígitos)",
    "municipio":           "nome do município (ex.: 'Campinas', 'São Paulo')",
    "uf":                  "sigla da UF (ex.: 'SP', 'BA')",
    "regiao":              "Norte, Nordeste, Centro-Oeste, Sudeste ou Sul",
    "populacao_2021":      "população residente estimada em 2021 (IBGE)",
    "pib_2021_mil_reais":  "PIB municipal de 2021 em MIL reais (IBGE)",
    "n_participantes":     "candidatos do ENEM 2023 considerados na média do município",
    "media_cn":            "média da nota de Ciências da Natureza (0 a 1000)",
    "media_ch":            "média da nota de Ciências Humanas (0 a 1000)",
    "media_lc":            "média da nota de Linguagens e Códigos (0 a 1000)",
    "media_mt":            "média da nota de Matemática (0 a 1000)",
    "media_redacao":       "média da nota de Redação (0 a 1000)",
    "media_geral":         "média das cinco notas por candidato, agregada por município",
    "pct_escola_publica":  "percentual de participantes de escola pública (0 a 100)",
    "pib_per_capita_2021": "PIB por habitante em 2021, em reais",
}

def descrever_esquema(df: pd.DataFrame) -> str:
    linhas = [f"- {c} ({df[c].dtype}): {DESCRICOES[c]}" for c in df.columns]
    return "\n".join(linhas)

ESQUEMA = descrever_esquema(df)
print(ESQUEMA)

- co_municipio (int64): código IBGE do município (7 dígitos)
- municipio (str): nome do município (ex.: 'Campinas', 'São Paulo')
- uf (str): sigla da UF (ex.: 'SP', 'BA')
- regiao (str): Norte, Nordeste, Centro-Oeste, Sudeste ou Sul
- populacao_2021 (float64): população residente estimada em 2021 (IBGE)
- pib_2021_mil_reais (float64): PIB municipal de 2021 em MIL reais (IBGE)
- n_participantes (int64): candidatos do ENEM 2023 considerados na média do município
- media_cn (float64): média da nota de Ciências da Natureza (0 a 1000)
- media_ch (float64): média da nota de Ciências Humanas (0 a 1000)
- media_lc (float64): média da nota de Linguagens e Códigos (0 a 1000)
- media_mt (float64): média da nota de Matemática (0 a 1000)
- media_redacao (float64): média da nota de Redação (0 a 1000)
- media_geral (float64): média das cinco notas por candidato, agregada por município
- pct_escola_publica (float64): percentual de participantes de escola pública (0 a 100)
- pib_per_capita_2021 (float6

---

# 13. Implementação do baseline

## Desenho

```
pergunta ─► [LLM, 1 chamada] ─► plano estruturado ─► [guarda de segurança] ─► [eval pandas] ─► [formatação] ─► resposta
                                   viavel?             tokens proibidos        determinístico  determinística
                                   codigo_pandas
                                   template_resposta
```

A única chamada ao LLM é usada para gerar o plano de execução: o modelo determina se a pergunta pode ser respondida, qual expressão pandas deve ser usada e qual frase será utilizada para apresentar o resultado. A partir daí, todo o restante do processo é determinístico.

Essa escolha traz duas consequências importantes, que são intencionais no desenho atual:

- o modelo gera a frase antes de conhecer o resultado numérico. Por isso, a resposta funciona como um template, com o marcador {resultado}. Essa é justamente uma das limitações que o agente Sintetizador do Entregável 3 deverá resolver;
- não existe validação ou retry. Se o código gerado estiver incorreto, o sistema pode produzir um resultado incorreto, como mostrado na seção 17. Essa limitação é importante para que seja possível medir de forma clara o ganho proporcionado pelo Validador do Entregável 2.

## Segurança da execução (RNF-06)

Antes de ser executado, o código gerado passa por uma verificação sintática. O uso de `ast.parse(mode="eval")` garante que o código seja composto por uma única expressão. Com isso, construções como atribuições, import e uso de ponto e vírgula são rejeitadas como erros de sintaxe.

Em seguida, a árvore sintática é percorrida para verificar se o código utiliza apenas os nomes permitidos (df, pd, np e uma lista restrita de builtins) e para bloquear o acesso a atributos privados. A expressão é então executada com eval, utilizando uma versão restrita de __builtins__.

É importante destacar que isso não constitui um sandbox de execução. O código continua sendo executado no mesmo processo, sem timeout e sem limite de memória. Portanto, por exemplo, um laço infinito pode travar o notebook.

O isolamento efetivo — utilizando subprocesso, timeout, bloqueio de acesso à rede e ao sistema de arquivos — será responsabilidade do Entregável 2 e está registrado como uma limitação na seção 18.


In [8]:
class PlanoConsulta(BaseModel):
    """O que a única chamada ao LLM precisa devolver."""
    viavel: bool = Field(
        description="True se a pergunta pode ser respondida SOMENTE com as colunas listadas.")
    motivo: str = Field(
        description="Se viavel=False, qual informação falta. Se viavel=True, a estratégia em uma frase.")
    codigo_pandas: str = Field(
        description="UMA expressão pandas sobre o DataFrame `df`. String vazia se viavel=False.")
    template_resposta: str = Field(
        description="Frase em português contendo o marcador {resultado}. Vazia se viavel=False.")
    colunas_usadas: list[str] = Field(
        description="Colunas do esquema usadas na expressão.")

structured_llm = llm.with_structured_output(PlanoConsulta, method="json_schema", include_raw=True)
print("[done]")

[done]


In [9]:
INSTRUCAO = """
Você traduz perguntas em português para consultas pandas sobre um DataFrame chamado `df`,
já carregado, com uma linha por município brasileiro.

COLUNAS DISPONÍVEIS (são as ÚNICAS existentes):
{esquema}

O RECORTE DOS DADOS:
- ENEM de 2023 apenas; indicadores do IBGE de 2021 apenas.
- Só entram candidatos que declararam escola, estiveram presentes nos dois dias e têm as
  cinco notas: 721.429 participantes em 5.481 municípios.
- O vínculo municipal é o município DA ESCOLA, não o de residência nem o de prova.

REGRAS:
1. Se a pergunta exigir qualquer informação que não esteja nas colunas acima — outro ano,
   outro exame, outro indicador (IDH, renda familiar, número de escolas), nota por
   disciplina específica (inglês, física, química), dado por candidato — responda com
   viavel=false, codigo_pandas="" e explique em `motivo` exatamente o que falta.
   NUNCA invente um número e nunca aproxime usando uma coluna diferente da pedida.
2. Se a pergunta for respondível, `codigo_pandas` deve ser UMA ÚNICA EXPRESSÃO Python
   (sem `import`, sem atribuição, sem `print`, sem ponto e vírgula) avaliada sobre `df`.
   Os únicos nomes disponíveis são `df`, `pd`, `np` e as funções round, len, sorted, list,
   min, max, sum, abs, float, int, str. Qualquer outro nome faz a execução ser recusada.
3. Municípios homônimos existem: filtre também por `uf` quando a pergunta citar o estado.
   "São Paulo" como município é `(df.municipio == "São Paulo") & (df.uf == "SP")`;
   "estado de São Paulo" é `df.uf == "SP"`.
4. Respeite filtros de robustez pedidos na pergunta (ex.: "com pelo menos 100
   participantes" vira `df.n_participantes >= 100`). Não invente filtros que não foram pedidos.
5. Para valor único devolva um escalar; para ranking devolva um DataFrame com as colunas
   relevantes já ordenado e limitado (`.nlargest`, `.head`); para comparação entre grupos
   devolva uma Series indexada pelo grupo.
6. `template_resposta` é uma frase em português que apresenta o resultado e contém
   EXATAMENTE UMA VEZ o marcador {{resultado}}. Não escreva o número você mesmo — você
   ainda não o conhece. Não use chaves para mais nada.

EXEMPLOS (formato, não conteúdo):

Pergunta: "Quantos municípios da Bahia estão na base?"
  viavel=true
  codigo_pandas: int((df.uf == "BA").sum())
  template_resposta: "A base tem {{resultado}} municípios da Bahia."

Pergunta: "Qual o número de professores por aluno em Natal?"
  viavel=false
  motivo: "O recorte não tem dados de docentes; as colunas cobrem notas do ENEM 2023,
           população e PIB municipais."
  codigo_pandas: ""
"""

def montar_prompt(pergunta: str) -> str:
    return INSTRUCAO.format(esquema=ESQUEMA) + f"\n\nPERGUNTA:\n{pergunta}\n"

print(montar_prompt("Qual é a média geral em Campinas?")[:400], "...")


Você traduz perguntas em português para consultas pandas sobre um DataFrame chamado `df`,
já carregado, com uma linha por município brasileiro.

COLUNAS DISPONÍVEIS (são as ÚNICAS existentes):
- co_municipio (int64): código IBGE do município (7 dígitos)
- municipio (str): nome do município (ex.: 'Campinas', 'São Paulo')
- uf (str): sigla da UF (ex.: 'SP', 'BA')
- regiao (str): Norte, Nordeste, Ce ...


In [10]:
# RNF-06: guarda sintática
NOMES_PERMITIDOS = {"df", "pd", "np"}
BUILTINS_PERMITIDOS = {
    "round": round, "len": len, "sorted": sorted, "list": list, "min": min,
    "max": max, "sum": sum, "abs": abs, "float": float, "int": int, "str": str,
    "dict": dict, "set": set, "zip": zip, "range": range, "bool": bool,
    "enumerate": enumerate, "True": True, "False": False, "None": None,
}

def codigo_seguro(codigo: str) -> list[str]:
    """Devolve os motivos de recusa. Lista vazia = liberado.

    `ast.parse(mode="eval")` já garante UMA expressão: atribuição, `import`,
    `print` e ponto e vírgula viram erro de sintaxe. Sobra checar quais nomes a
    expressão toca, daí o passeio pela árvore.
    """
    try:
        arvore = ast.parse(codigo, mode="eval")
    except SyntaxError as e:
        return [f"não é uma expressão única: {e.msg}"]

    permitidos = NOMES_PERMITIDOS | set(BUILTINS_PERMITIDOS)
    motivos = []
    for no in ast.walk(arvore):
        if isinstance(no, ast.Name) and no.id not in permitidos:
            motivos.append(f"nome não permitido: {no.id}")
        if isinstance(no, ast.Attribute) and no.attr.startswith("_"):
            motivos.append(f"atributo privado: {no.attr}")
    return motivos

def executar(codigo: str):
    """Executa a expressão em espaço de nomes restrito. Devolve (valor, erro)."""
    motivos = codigo_seguro(codigo)
    if motivos:
        return None, f"código bloqueado pela guarda de segurança: {motivos}"
    try:
        return eval(codigo, {"__builtins__": BUILTINS_PERMITIDOS},
                    {"df": df, "pd": pd, "np": np}), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

print("[done]")

[done]


## Guarda

Sem LLM: expressões legítimas passam, as demais são recusadas antes de qualquer execução.


In [11]:
# RNF-06: a guarda recusa antes de executar
for tentativa in [
    'df.media_geral.mean()',                                              # legítima
    'df.sort_values("media_mt", ascending=False).head(3)["municipio"]',   # kwargs: permitido
    '__import__("os").system("ls")',                                      # nome não permitido
    'df.to_csv("dados.csv")',                                             # nome permitido, mas escreve em disco
    'x = df.media_geral.mean()',                                          # não é expressão única
    'df.__class__.__mro__',                                               # atributo privado
]:
    motivos = codigo_seguro(tentativa)
    print(f'{"LIBERA" if not motivos else "RECUSA"}  {tentativa}')
    if motivos:
        print(f'         {motivos}')

LIBERA  df.media_geral.mean()
LIBERA  df.sort_values("media_mt", ascending=False).head(3)["municipio"]
RECUSA  __import__("os").system("ls")
         ['nome não permitido: __import__']
LIBERA  df.to_csv("dados.csv")
RECUSA  x = df.media_geral.mean()
         ['não é uma expressão única: invalid syntax']
RECUSA  df.__class__.__mro__
         ['atributo privado: __mro__', 'atributo privado: __class__']


In [12]:
NOTA_RECORTE = (
    "Recorte: ENEM 2023, apenas candidatos com escola declarada, presentes nos dois dias "
    "e com as cinco notas (721.429 de ~3,9 milhões de inscritos), agregados pelo município "
    "da escola. Indicadores do IBGE são de 2021."
)

def formatar_valor(valor) -> str:
    """Converte o resultado bruto da execução em texto legível (determinístico)."""
    if valor is None:
        return "sem resultado"
    if isinstance(valor, (bool, np.bool_)):
        return "sim" if valor else "não"
    if isinstance(valor, (int, np.integer)):
        return str(int(valor))
    if isinstance(valor, (float, np.floating)):
        # correlações e proporções perdem sentido com duas casas
        casas = 4 if abs(float(valor)) < 1 else 2
        texto = f"{float(valor):.{casas}f}"
        return texto.rstrip("0").rstrip(".") if "." in texto else texto
    if isinstance(valor, pd.Series):
        return "; ".join(f"{i}: {formatar_valor(v)}" for i, v in valor.head(10).items())
    if isinstance(valor, pd.DataFrame):
        rotulos = [c for c in valor.columns if valor[c].dtype == object or
                   pd.api.types.is_string_dtype(valor[c])]
        numeros = [c for c in valor.columns if c not in rotulos and c != "co_municipio"]
        linhas = []
        for _, linha in valor.head(10).iterrows():
            nome = " / ".join(str(linha[c]) for c in rotulos)
            medidas = ", ".join(f"{c}: {formatar_valor(linha[c])}" for c in numeros)
            linhas.append(f"{nome} ({medidas})" if nome and medidas else (nome or medidas))
        return "; ".join(linhas)
    return str(valor)

class Resposta(BaseModel):
    texto: str
    resultado: Any = None
    codigo: str = ""
    viavel: bool = True
    motivo: str = ""
    erro_execucao: Optional[str] = None

def baseline(pergunta: str) -> tuple[Resposta, dict]:
    """Baseline: uma chamada ao LLM + execução e formatação determinísticas."""
    inicio = time.perf_counter()
    try:
        saida = structured_llm.invoke(montar_prompt(pergunta))
    except Exception as e:                              # erro de API conta como falha, não derruba o notebook
        latencia = time.perf_counter() - inicio
        erro = f"erro na chamada ao modelo: {type(e).__name__}: {str(e)[:200]}"
        return Resposta(texto=f"Falha na chamada ao modelo. {erro}", erro_execucao=erro), {
            "latencia_s": round(latencia, 2), "tokens_entrada": None, "tokens_saida": None,
            "chamadas_llm": 1, "erro_parse": erro}
    latencia = time.perf_counter() - inicio

    uso = getattr(saida["raw"], "usage_metadata", None) or {}
    metricas = {
        "latencia_s": round(latencia, 2),
        "tokens_entrada": uso.get("input_tokens"),
        "tokens_saida": uso.get("output_tokens"),
        "chamadas_llm": 1,
        "erro_parse": str(saida["parsing_error"]) if saida["parsing_error"] else None,
    }

    plano = saida["parsed"]
    if plano is None:                                   # RNF-01 violado
        return Resposta(texto="Falha ao interpretar a saída do modelo.",
                        viavel=False, motivo=str(metricas["erro_parse"])), metricas

    if not plano.viavel:                                # RF-05
        return Resposta(texto=f"Não é possível responder com os dados disponíveis. {plano.motivo}",
                        viavel=False, motivo=plano.motivo), metricas

    valor, erro = executar(plano.codigo_pandas)
    if erro:
        return Resposta(texto=f"A consulta gerada não pôde ser executada. {erro}",
                        codigo=plano.codigo_pandas, motivo=plano.motivo,
                        erro_execucao=erro), metricas

    texto_valor = formatar_valor(valor)
    try:
        frase = plano.template_resposta.format(resultado=texto_valor)
    except (KeyError, IndexError, ValueError):          # template malformado
        frase = f"{plano.template_resposta} {texto_valor}".strip()

    return Resposta(texto=f"{frase}\n\n{NOTA_RECORTE}", resultado=valor,
                    codigo=plano.codigo_pandas, motivo=plano.motivo), metricas

print("[done]")

[done]


## Primeira execução — uma pergunta viável e outra inviável

As duas perguntas apresentadas a seguir **não fazem parte do conjunto de avaliação**. Elas são usadas apenas para ilustrar o comportamento do sistema antes de definir e congelar os critérios de avaliação.


In [13]:
for pergunta in ["Qual é a média de redação no estado do Ceará?",
                 "Quantas escolas particulares existem em Fortaleza?"]:
    r, m = baseline(pergunta)
    print("=" * 78)
    print("PERGUNTA :", pergunta)
    print("VIÁVEL   :", r.viavel)
    print("CÓDIGO   :", r.codigo or "—")
    print("RESPOSTA :", r.texto.split("\n")[0])
    print("MÉTRICAS :", m)

PERGUNTA : Qual é a média de redação no estado do Ceará?
VIÁVEL   : True
CÓDIGO   : df.loc[df.uf == 'CE', 'media_redacao'].mean()
RESPOSTA : A média de redação no estado do Ceará é 531.47.
MÉTRICAS : {'latencia_s': 0.88, 'tokens_entrada': 1327, 'tokens_saida': 352, 'chamadas_llm': 1, 'erro_parse': None}
PERGUNTA : Quantas escolas particulares existem em Fortaleza?
VIÁVEL   : False
CÓDIGO   : —
RESPOSTA : Não é possível responder com os dados disponíveis. O conjunto de dados não contém informações sobre o número de escolas particulares; as colunas disponíveis referem-se apenas a dados demográficos, PIB e médias de notas do ENEM 2023.
MÉTRICAS : {'latencia_s': 0.56, 'tokens_entrada': 1323, 'tokens_saida': 310, 'chamadas_llm': 1, 'erro_parse': None}


# 14. Conjunto de avaliação

O conjunto de avaliação é composto por **13 casos**, sendo 11 avaliados automaticamente e 2 avaliados por meio de uma rubrica manual. Para cada caso, a resposta de referência foi calculada previamente com uma consulta pandas escrita manualmente sobre o mesmo artefato. Esse cálculo foi feito **antes de qualquer chamada ao LLM e antes de qualquer ajuste no prompt**, garantindo que a referência não fosse influenciada pelo comportamento do sistema.

O conjunto está **congelado**: os mesmos 13 casos serão utilizados nos Entregáveis 2, 3 e 4. Caso algum caso seja alterado, será necessário reexecutar o baseline, pois a comparação entre as versões só é válida quando todas são avaliadas usando a mesma referência.

| ID  | Pergunta                                                       | Tipo                                    | Referência                                                        | Verificação                     |
| --- | -------------------------------------------------------------- | --------------------------------------- | ----------------------------------------------------------------- | ------------------------------- |
| T01 | Média geral em Campinas (SP)                                   | normal / valor único                    | 584,24                                                            | automática (numérica, 1%)       |
| T02 | Quantos municípios do Acre há na base                          | normal / contagem                       | 22                                                                | automática (numérica, exata)    |
| T03 | Top 5 de SP em matemática, com ≥ 100 participantes             | ranking com filtro                      | Valinhos, São João da Boa Vista, Amparo, São José dos Campos, Jaú | automática (cobertura de lista) |
| T04 | Região com maior média de redação                              | agregação por grupo                     | Sudeste                                                           | automática (nome na resposta)   |
| T05 | Quantos municípios têm média geral acima de 550                | filtro + contagem                       | 990                                                               | automática (numérica, exata)    |
| T06 | Correlação PIB per capita × média geral (≥ 50 participantes)   | **cruzamento ENEM × IBGE**              | 0,2868                                                            | automática (numérica, 5%)       |
| T07 | Entre os 10 mais populosos, qual tem maior média de matemática | **cruzamento + composto**               | Belo Horizonte (638,58)                                           | automática (nome + valor)       |
| T08 | Média geral do Nordeste × média geral do Sul                   | comparação entre grupos                 | 488,64 e 528,01                                                   | automática (dois valores)       |
| T09 | Nota média de **inglês** em Salvador                           | informação ausente                      | abstenção                                                         | automática (abstenção)          |
| T10 | **IDH** de Recife                                              | informação ausente                      | abstenção                                                         | automática (abstenção)          |
| T11 | Média geral do ENEM **2019** em Belo Horizonte                 | fora do recorte temporal                | abstenção                                                         | automática (abstenção)          |
| T12 | "Qual é o melhor município para estudar?"                      | ambíguo                                 | —                                                                 | **manual**                      |
| T13 | "Qual é a média de São Paulo?"                                 | ambíguo (capital × estado; qual média?) | —                                                                 | **manual**                      |

## Como os tipos de caso foram escolhidos

Os casos foram escolhidos para cobrir diferentes níveis de dificuldade e situações que o sistema precisa saber tratar.

* **T01–T05** representam consultas relativamente comuns: obtenção de um valor único, contagem, ranking com filtro de robustez e agregação por grupo.

* **T06–T08** representam os casos mais específicos do projeto. Eles exigem o cruzamento entre os dados do ENEM e do IBGE ou a combinação de mais de uma operação. No T07, por exemplo, é necessário primeiro identificar os 10 municípios mais populosos e, em seguida, encontrar entre eles aquele com a maior média de matemática.

* **T09–T11** foram incluídos como casos de erro ou de informação indisponível. Cada um testa uma situação diferente: uma coluna que não existe na base (inglês), um indicador que não está disponível (IDH) e um ano fora do recorte temporal (2019). Este último caso merece atenção especial, porque o modelo pode ter informações sobre o ENEM de 2019 em sua memória paramétrica e acabar respondendo com base nesse conhecimento, mesmo sem consultar o `DataFrame`.

* **T12–T13** são perguntas ambíguas e, portanto, não possuem uma resposta única definida. Esses casos não devem ser tratados simplesmente como erros. Eles servem para avaliar se, ao longo dos entregáveis, o sistema passa a identificar a ambiguidade e pedir esclarecimentos ao usuário (RF-09), em vez de escolher uma interpretação sem avisar.

> Com 11 casos avaliados automaticamente, cada acerto corresponde a aproximadamente 9 pontos percentuais. Esse número ainda é pequeno demais para sustentar uma conclusão sobre superioridade estatística entre diferentes arquiteturas. Por isso, conforme previsto no planejamento do projeto, o conjunto de avaliação será ampliado para cerca de 25–30 casos no Entregável 4.



In [14]:
# Congelado em 07/09/2026. As referências foram calculadas com consultas pandas
# escritas à mão sobre este mesmo artefato, antes de qualquer chamada ao LLM.
test_cases = [
    {"id": "T01", "tipo": "normal", "verificacao": "auto", "criterio": "numerico",
     "pergunta": "Qual é a média geral do ENEM 2023 no município de Campinas, em São Paulo?",
     "esperado": 584.24, "tolerancia": 0.01,
     "referencia": 'df.loc[(df.municipio == "Campinas") & (df.uf == "SP"), "media_geral"].item()'},

    {"id": "T02", "tipo": "normal", "verificacao": "auto", "criterio": "numerico",
     "pergunta": "Quantos municípios do estado do Acre estão na base?",
     "esperado": 22, "tolerancia": 0.0,
     "referencia": 'int((df.uf == "AC").sum())'},

    {"id": "T03", "tipo": "ranking", "verificacao": "auto", "criterio": "lista",
     "pergunta": ("Quais são os 5 municípios de São Paulo com maior média em matemática, "
                  "considerando apenas municípios com pelo menos 100 participantes?"),
     "esperado": ["Valinhos", "São João da Boa Vista", "Amparo",
                  "São José dos Campos", "Jaú"],
     "cobertura_minima": 1.0,
     "referencia": 'df[(df.uf == "SP") & (df.n_participantes >= 100)].nlargest(5, "media_mt")'},

    {"id": "T04", "tipo": "agregação por grupo", "verificacao": "auto", "criterio": "lista",
     "pergunta": "Qual região do país tem a maior média de redação?",
     "esperado": ["Sudeste"], "cobertura_minima": 1.0,
     "referencia": 'df.groupby("regiao")["media_redacao"].mean().idxmax()'},

    {"id": "T05", "tipo": "normal", "verificacao": "auto", "criterio": "numerico",
     "pergunta": "Quantos municípios têm média geral acima de 550?",
     "esperado": 990, "tolerancia": 0.0,
     "referencia": 'int((df.media_geral > 550).sum())'},

    {"id": "T06", "tipo": "cruzamento ENEM×IBGE", "verificacao": "auto", "criterio": "numerico",
     "pergunta": ("Qual é a correlação entre o PIB per capita de 2021 e a média geral do ENEM "
                  "dos municípios com pelo menos 50 participantes?"),
     "esperado": 0.2868, "tolerancia": 0.05,
     "referencia": ('df[df.n_participantes >= 50]["pib_per_capita_2021"]'
                    '.corr(df[df.n_participantes >= 50]["media_geral"])')},

    {"id": "T07", "tipo": "cruzamento + composto", "verificacao": "auto", "criterio": "misto",
     "pergunta": ("Entre os 10 municípios mais populosos do país, qual tem a maior média "
                  "em matemática?"),
     "esperado": 638.58, "tolerancia": 0.01, "esperado_texto": ["Belo Horizonte"],
     "referencia": 'df.nlargest(10, "populacao_2021").nlargest(1, "media_mt")'},

    {"id": "T08", "tipo": "comparação entre grupos", "verificacao": "auto", "criterio": "lista",
     "pergunta": ("Compare a média geral dos municípios do Nordeste com a dos municípios "
                  "do Sul."),
     "esperado": ["488", "528"], "cobertura_minima": 1.0,
     "referencia": 'df.groupby("regiao")["media_geral"].mean()[["Nordeste", "Sul"]]'},

    {"id": "T09", "tipo": "informação ausente", "verificacao": "auto", "criterio": "abstencao",
     "pergunta": "Qual é a nota média de inglês no ENEM 2023 em Salvador?",
     "esperado": None,
     "referencia": "não existe nota por língua estrangeira no artefato"},

    {"id": "T10", "tipo": "informação ausente", "verificacao": "auto", "criterio": "abstencao",
     "pergunta": "Qual é o IDH de Recife?",
     "esperado": None,
     "referencia": "IDH não é indicador do artefato (só população, PIB e PIB per capita)"},

    {"id": "T11", "tipo": "fora do recorte temporal", "verificacao": "auto", "criterio": "abstencao",
     "pergunta": "Qual foi a média geral do ENEM 2019 em Belo Horizonte?",
     "esperado": None,
     "referencia": "o artefato só tem 2023; o risco é o modelo responder de memória"},

    {"id": "T12", "tipo": "ambíguo", "verificacao": "manual", "criterio": "manual",
     "pergunta": "Qual é o melhor município para estudar?",
     "esperado": None,
     "nota": ("Boa resposta: apontar que 'melhor' não está definido, oferecer um critério "
              "(ex.: maior media_geral com n_participantes mínimo) e declarar a escolha. "
              "Resposta ruim: devolver um município sem dizer sob qual critério.")},

    {"id": "T13", "tipo": "ambíguo", "verificacao": "manual", "criterio": "manual",
     "pergunta": "Qual é a média de São Paulo?",
     "esperado": None,
     "nota": ("Duplamente ambíguo: capital ou estado? média de qual área? Boa resposta: "
              "pedir esclarecimento ou responder declarando explicitamente a leitura "
              "adotada (ex.: 'município de São Paulo, média geral').")},
]

print(len(test_cases), "casos;",
      sum(c["verificacao"] == "auto" for c in test_cases), "automáticos;",
      sum(c["verificacao"] == "manual" for c in test_cases), "manuais.")

13 casos; 11 automáticos; 2 manuais.


## Sanidade das referências

A célula abaixo reexecuta as consultas de referência diretamente sobre o artefato carregado. Ela não utiliza o LLM e serve apenas para confirmar que os valores apresentados na tabela acima continuam válidos para o arquivo de dados atual.

Se algum resultado diferir do valor de referência, isso indica que o artefato foi alterado. Nesse caso, o baseline precisa ser reexecutado para que as referências sejam atualizadas.



In [15]:
conferencia = []
for caso in test_cases:
    if caso["criterio"] in ("abstencao", "manual"):
        continue
    valor, erro = executar(caso["referencia"])
    conferencia.append({"id": caso["id"], "referencia_executada": formatar_valor(valor),
                        "erro": erro})

pd.DataFrame(conferencia)

,id,referencia_executada,erro
0,T01,584.24,None
1,T02,22,None
2,T03,Valinhos / SP / Sudeste (populacao_2021: 13316...,None
3,T04,Sudeste,None
4,T05,990,None
5,T06,0.2868,None
6,T07,Belo Horizonte / MG / Sudeste (populacao_2021:...,None
7,T08,Nordeste: 488.64; Sul: 528.01,None


# 15. Implementação da verificação

São utilizadas cinco verificações, cada uma associada a um dos requisitos apresentados na seção 6:

| Função              | Requisito    | O que mede                                                                                           |
| ------------------- | ------------ | ---------------------------------------------------------------------------------------------------- |
| `resultado_correto` | RF-01, RF-03 | verifica se o **valor obtido pela execução** corresponde à referência, dentro da tolerância definida |
| `cobertura_lista`   | RF-02        | verifica qual proporção dos itens de referência aparece na resposta final                            |
| `texto_fiel`        | RF-04        | verifica se o número apresentado ao usuário corresponde ao valor retornado pela execução             |
| `abstencao_valida`  | RF-05        | verifica se o sistema recusou a consulta, não executou código e informou o que está faltando         |
| `tem_evidencia`     | RF-06        | verifica se o código executado e o resultado bruto estão disponíveis para auditoria                  |

O ponto principal é que a **correção é avaliada a partir do resultado da execução**, e não pela presença de determinada palavra ou valor no texto da resposta. Dessa forma, a verificação não é afetada por paráfrases ou diferenças na forma de apresentar o resultado. Isso é possível porque o baseline disponibiliza o valor bruto produzido pela execução.



In [16]:
def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip()

# Números aparecem em três formatos no notebook: pt-BR ("1.234,56"), inglês
# ("1,234.56") e simples ("584.24"). Confundi-los reprova resposta certa.
# `formatar_valor` emite decimal com ponto, então um ponto só é separador de 
# milhar quando há DOIS ou mais grupos ("1.234.567") ou quando uma vírgula 
# decimal vem depois ("1.234,56").
PT_BR_MILHAR = r"-?\d{1,3}(?:\.\d{3}){2,}(?:,\d+)?|-?\d{1,3}(?:\.\d{3})+,\d+"
EN_MILHAR = r"-?\d{1,3}(?:,\d{3})+(?:\.\d+)?"
PADRAO_NUMERO = re.compile(
    PT_BR_MILHAR                             # 1.234.567 / 1.234,56  (pt-BR)
    + "|" + EN_MILHAR                        # 1,234,567.89          (inglês)
    + r"|-?\d+(?:[.,]\d+)?"                  # 584.24, 584,24, 0.2868
)

def _para_float(texto: str) -> float:
    if re.fullmatch(PT_BR_MILHAR, texto):
        return float(texto.replace(".", "").replace(",", "."))
    if re.fullmatch(EN_MILHAR, texto):
        return float(texto.replace(",", ""))
    return float(texto.replace(",", "."))

def numeros_de(valor) -> list[float]:
    """Todos os números contidos em um escalar, Series, DataFrame ou texto."""
    if isinstance(valor, (bool, np.bool_)):
        return []
    if isinstance(valor, (int, float, np.integer, np.floating)):
        return [float(valor)]
    if isinstance(valor, pd.Series):
        return [float(v) for v in pd.to_numeric(valor, errors="coerce").dropna()]
    if isinstance(valor, pd.DataFrame):
        numericas = valor.select_dtypes("number")
        return [float(v) for v in numericas.to_numpy().ravel() if pd.notna(v)]
    return [_para_float(t) for t in PADRAO_NUMERO.findall(str(valor))]

def proximo(obtido: float, esperado: float, tolerancia: float) -> bool:
    return abs(obtido - esperado) <= tolerancia * max(abs(esperado), 1.0)

def resultado_correto(caso: dict, resposta) -> bool:
    """RF-01/RF-03: o valor executado contém a referência dentro da tolerância."""
    if resposta.resultado is None:
        return False
    return any(proximo(n, caso["esperado"], caso.get("tolerancia", 0.01))
               for n in numeros_de(resposta.resultado))

def cobertura_lista(caso: dict, resposta) -> float:
    """RF-02: fração dos itens de referência presentes no texto final."""
    texto = normalizar(resposta.texto)
    itens = caso["esperado"]
    return sum(normalizar(i) in texto for i in itens) / len(itens)

def texto_fiel(resposta) -> bool:
    """RF-04: o número mostrado ao usuário é um número devolvido pela execução."""
    if resposta.resultado is None:
        return False
    do_resultado = numeros_de(resposta.resultado)
    if not do_resultado:
        return True                      # resultado não numérico (ex.: nome de região)
    do_texto = numeros_de(resposta.texto.split("Recorte:")[0])
    return any(proximo(t, r, 0.011) for r in do_resultado for t in do_texto)

MARCADORES_AUSENCIA = [
    "nao e possivel", "nao esta", "nao consta", "nao ha", "nao existe",
    "nao contem", "nao dispon", "nao inclui", "nao possui", "sem dados",
    "fora do recorte", "nao coberto", "nao ha coluna", "nao foi encontrad",
]

def abstencao_valida(resposta) -> bool:
    """RF-05: não basta dizer 'não'; não pode ter executado código nem dado número."""
    if resposta.viavel or resposta.codigo or resposta.resultado is not None:
        return False
    return any(m in normalizar(resposta.texto) for m in MARCADORES_AUSENCIA)

def tem_evidencia(resposta) -> bool:
    """RF-06: resposta viável precisa expor o código e o resultado."""
    return bool(resposta.codigo) and resposta.resultado is not None

def avaliar(caso: dict, resposta) -> dict:
    """Devolve o veredito e os componentes que o formaram."""
    criterio = caso["criterio"]

    if criterio == "manual":
        return {"aprovado": None, "cobertura": None, "detalhe": "rubrica manual"}

    if criterio == "abstencao":
        ok = abstencao_valida(resposta)
        return {"aprovado": ok, "cobertura": None,
                "detalhe": "abstenção válida" if ok else "deveria ter se abstido"}

    if criterio == "lista":
        cob = cobertura_lista(caso, resposta)
        ok = cob >= caso.get("cobertura_minima", 1.0)
        return {"aprovado": bool(ok), "cobertura": round(cob, 2),
                "detalhe": f"cobertura {cob:.0%}"}

    # numérico e misto: o valor precisa bater e o texto precisa refletir o valor.
    # no misto o resultado pode ser um nome (sem números): só o nome é cobrado.
    fiel = texto_fiel(resposta)
    nome_ok = all(normalizar(t) in normalizar(resposta.texto)
                  for t in caso.get("esperado_texto", []))
    tem_numeros = bool(numeros_de(resposta.resultado)) if resposta.resultado is not None else False
    if criterio == "misto" and not tem_numeros:
        correto = resposta.resultado is not None
    else:
        correto = resultado_correto(caso, resposta)

    ok = correto and fiel and nome_ok
    return {"aprovado": bool(ok), "cobertura": None,
            "detalhe": (f"valor={'ok' if correto else 'errado'}, "
                        f"texto_fiel={'ok' if fiel else 'nao'}"
                        + ("" if nome_ok else ", nome esperado ausente"))}

print("[done]")

[done]


## Verificação ingênua

É tentador verificar a abstenção simplesmente procurando a palavra `"não"` na resposta e verificar a correção procurando o número esperado no texto. A célula abaixo mostra por que essas duas abordagens, sozinhas, não são suficientes. E essa demonstração não depende do LLM: as respostas utilizadas no teste são construídas manualmente.


In [17]:
# (a) Nega uma coisa e inventa outra: a palavra "não" está presente, mas houve execução
# e um número foi entregue ao usuário.
resposta_falsa_abstencao = Resposta(
    texto="Não há coluna de IDH, mas o IDH de Recife é 0,772.",
    resultado=0.772, codigo='df.media_geral.mean()', viavel=True,
)

# (b) O número certo aparece no texto, mas veio do modelo, não da execução.
resposta_numero_alucinado = Resposta(
    texto="A média geral de Campinas é 584,24.",
    resultado=498.10, codigo='df.media_geral.mean()', viavel=True,
)

print("(a) verificação ingênua ('não' no texto) :", "nao" in normalizar(resposta_falsa_abstencao.texto))
print("(a) abstencao_valida                     :", abstencao_valida(resposta_falsa_abstencao))
print()
print("(b) verificação ingênua ('584,24' no texto):",
      "584,24" in resposta_numero_alucinado.texto)
print("(b) resultado_correto (contra a referência):",
      resultado_correto(test_cases[0], resposta_numero_alucinado))
print("(b) texto_fiel (texto × execução)         :", texto_fiel(resposta_numero_alucinado))

(a) verificação ingênua ('não' no texto) : True
(a) abstencao_valida                     : False

(b) verificação ingênua ('584,24' no texto): True
(b) resultado_correto (contra a referência): False
(b) texto_fiel (texto × execução)         : False


A verificação ingênua **aprova os dois casos**, enquanto as verificações implementadas reprovam ambos. No primeiro caso, porque houve execução e o sistema apresentou um número quando deveria ter se abstido de responder. No segundo, porque o número apresentado na resposta não corresponde ao valor produzido pelo código executado.

Essa diferença é importante não apenas para este notebook. No Entregável 4, a comparação entre as diferentes arquiteturas será baseada justamente nesses resultados. Se o critério de avaliação estiver incorreto, podemos acabar com uma tabela aparentemente boa, mas que leva a uma conclusão errada.


# 16. Experimentos

O baseline é executado sobre os 13 casos congelados. Para cada caso, são registrados a resposta final, o código gerado, o resultado bruto da execução, o veredito da avaliação, a latência, o número de tokens utilizados e a quantidade de chamadas realizadas.

A variável `registros` é a base para o **log estruturado** previsto na proposta do projeto. Neste momento, cada entrada representa a única transição existente no fluxo atual: pergunta → plano → execução → texto. A partir do Entregável 2, cada transição entre agentes — incluindo as reprovações feitas pelo Validador — também será registrada seguindo esse mesmo formato.



In [18]:
registros = []

for caso in test_cases:
    resposta, metricas = baseline(caso["pergunta"])
    nota = avaliar(caso, resposta)

    registros.append({
        "id": caso["id"],
        "tipo": caso["tipo"],
        "pergunta": caso["pergunta"],
        "resposta": resposta.texto.split("\n")[0],
        "codigo": resposta.codigo,
        "resultado_bruto": formatar_valor(resposta.resultado),
        "viavel": resposta.viavel,
        "erro_execucao": resposta.erro_execucao,
        "evidencia": tem_evidencia(resposta) if resposta.viavel else None,
        **nota,
        **metricas,
    })

    print("=" * 78)
    print(f'[{caso["id"]}] ({caso["tipo"]}) {caso["pergunta"]}')
    print("CÓDIGO   :", resposta.codigo or "— (abstenção)")
    print("RESULTADO:", formatar_valor(resposta.resultado))
    print("RESPOSTA :", resposta.texto.split("\n")[0])
    print(f'VEREDITO : {nota["aprovado"]}  ({nota["detalhe"]})'
          f'  | {metricas["latencia_s"]} s')

[T01] (normal) Qual é a média geral do ENEM 2023 no município de Campinas, em São Paulo?
CÓDIGO   : df.loc[(df.municipio=="Campinas") & (df.uf=="SP"), "media_geral"].iloc[0]
RESULTADO: 584.24
RESPOSTA : A média geral do ENEM 2023 em Campinas, SP, é 584.24.
VEREDITO : True  (valor=ok, texto_fiel=ok)  | 1.26 s
[T02] (normal) Quantos municípios do estado do Acre estão na base?
CÓDIGO   : int((df.uf == "AC").sum())
RESULTADO: 22
RESPOSTA : A base tem 22 municípios do estado do Acre.
VEREDITO : True  (valor=ok, texto_fiel=ok)  | 0.69 s
[T03] (ranking) Quais são os 5 municípios de São Paulo com maior média em matemática, considerando apenas municípios com pelo menos 100 participantes?
CÓDIGO   : df[(df.uf == "SP") & (df.n_participantes >= 100)].nlargest(5, 'media_mt')[['municipio', 'media_mt']].reset_index(drop=True)
RESULTADO: Valinhos (media_mt: 647.63); São João da Boa Vista (media_mt: 640.98); Amparo (media_mt: 625.08); São José dos Campos (media_mt: 623.02); Jaú (media_mt: 613.55)
RESPO

# 17. Resultados


In [19]:
pd.set_option("display.max_colwidth", 60)

df_res = pd.DataFrame(registros)
df_res[["id", "tipo", "aprovado", "detalhe", "resultado_bruto",
        "latencia_s", "tokens_entrada", "tokens_saida"]]

,id,tipo,aprovado,detalhe,resultado_bruto,latencia_s,tokens_entrada,tokens_saida
0,T01,normal,True,"valor=ok, texto_fiel=ok",584.24,1.26,1335,682
1,T02,normal,True,"valor=ok, texto_fiel=ok",22,0.69,1326,221
2,T03,ranking,True,cobertura 100%,Valinhos (media_mt: 647.63); São João da Boa Vista (medi...,1.24,1341,818
3,T04,agregação por grupo,True,cobertura 100%,Sudeste,7.09,1327,501
4,T05,normal,True,"valor=ok, texto_fiel=ok",990,3.68,1326,225
5,T06,cruzamento ENEM×IBGE,True,"valor=ok, texto_fiel=ok",0.2868,5.16,1345,557
6,T07,cruzamento + composto,True,"valor=ok, texto_fiel=ok",Belo Horizonte,10.32,1335,879
7,T08,comparação entre grupos,True,cobertura 100%,Nordeste: 488.64; Sul: 528.01,9.49,1330,1053
8,T09,informação ausente,True,abstenção válida,sem resultado,15.84,1331,339
9,T10,informação ausente,True,abstenção válida,sem resultado,0.65,1323,158


In [20]:
# Código gerado em cada caso
for r in registros:
    print(f'[{r["id"]}] {r["codigo"] or "— (abstenção)"}')

[T01] df.loc[(df.municipio=="Campinas") & (df.uf=="SP"), "media_geral"].iloc[0]
[T02] int((df.uf == "AC").sum())
[T03] df[(df.uf == "SP") & (df.n_participantes >= 100)].nlargest(5, 'media_mt')[['municipio', 'media_mt']].reset_index(drop=True)
[T04] df.groupby('regiao')['media_redacao'].mean().idxmax()
[T05] int((df.media_geral > 550).sum())
[T06] df.loc[df.n_participantes>=50,'pib_per_capita_2021'].corr(df.loc[df.n_participantes>=50,'media_geral'])
[T07] df.nlargest(10, 'populacao_2021').sort_values('media_mt', ascending=False).iloc[0]['municipio']
[T08] df[df.regiao.isin(['Nordeste','Sul'])].groupby('regiao')['media_geral'].mean()
[T09] — (abstenção)
[T10] — (abstenção)
[T11] — (abstenção)
[T12] df.nlargest(1, 'media_geral')[['municipio', 'uf', 'media_geral']]
[T13] df.loc[(df.municipio == "São Paulo") & (df.uf == "SP"), "media_geral"].values[0]


In [21]:
# Preços por milhão de tokens (fonte: https://groq.com/pricing)
PRECO_USD_POR_MILHAO = {"entrada": 0.10, "saida": 0.50}

autos = df_res[df_res["aprovado"].notna()]
tokens_in = df_res["tokens_entrada"].fillna(0).sum()
tokens_out = df_res["tokens_saida"].fillna(0).sum()
custo = (tokens_in * PRECO_USD_POR_MILHAO["entrada"]
         + tokens_out * PRECO_USD_POR_MILHAO["saida"]) / 1e6

RESUMO = {
    "casos_totais": int(len(df_res)),
    "casos_automaticos": int(len(autos)),
    "taxa_aprovacao": round(float(autos["aprovado"].astype(bool).mean()), 3),
    "acertos": int(autos["aprovado"].astype(bool).sum()),
    "erros_execucao": int(df_res["erro_execucao"].notna().sum()),
    "erros_parse": int(df_res["erro_parse"].notna().sum()),
    "abstencoes": int((~df_res["viavel"].astype(bool)).sum()),
    "latencia_mediana_s": round(float(df_res["latencia_s"].median()), 2),
    "latencia_max_s": round(float(df_res["latencia_s"].max()), 2),
    "chamadas_llm": int(df_res["chamadas_llm"].sum()),
    "tokens_entrada": int(tokens_in),
    "tokens_saida": int(tokens_out),
    "custo_estimado_usd": round(float(custo), 6),
}
RESUMO

{'casos_totais': 13,
 'casos_automaticos': 11,
 'taxa_aprovacao': 1.0,
 'acertos': 11,
 'erros_execucao': 0,
 'erros_parse': 0,
 'abstencoes': 3,
 'latencia_mediana_s': 5.16,
 'latencia_max_s': 15.84,
 'chamadas_llm': 13,
 'tokens_entrada': 17295,
 'tokens_saida': 7005,
 'custo_estimado_usd': 0.005232}

In [22]:
# Análise de performance do baseline
(df_res[df_res["aprovado"].notna()]
 .assign(aprovado=lambda d: d["aprovado"].astype(bool))
 .groupby("tipo")
 .agg(casos=("aprovado", "size"),
      acertos=("aprovado", "sum"),
      latencia_mediana=("latencia_s", "median"))
 .assign(taxa=lambda d: (d["acertos"] / d["casos"]).round(2)))

,casos,acertos,latencia_mediana,taxa
tipo,,,,
agregação por grupo,1,1,7.090,1.0
comparação entre grupos,1,1,9.490,1.0
cruzamento + composto,1,1,10.320,1.0
cruzamento ENEM×IBGE,1,1,5.160,1.0
fora do recorte temporal,1,1,11.800,1.0
informação ausente,2,2,8.245,1.0
normal,3,3,1.260,1.0
ranking,1,1,1.240,1.0


In [23]:
referencia = {
    "run": RUN_INFO,
    "dados": {"origem": origem_dados, "linhas": int(df.shape[0]),
              "participantes": int(df["n_participantes"].sum())},
    "resumo": RESUMO,
    "registros": registros,
}
with open("baseline_v1_resultados.json", "w", encoding="utf-8") as f:
    json.dump(referencia, f, ensure_ascii=False, indent=2, default=str)

print("Salvo em baseline_v1_resultados.json")

Salvo em baseline_v1_resultados.json


## Rubrica manual — casos ambíguos

Os casos T12 e T13 não possuem uma única resposta correta. Por isso, a avaliação é feita manualmente, com uma nota de 0 a 2, seguindo o mesmo critério que será utilizado nos próximos entregáveis:

| Nota | Critério                                                                                        |
| ---- | ----------------------------------------------------------------------------------------------- |
| 0    | Escolheu uma interpretação e respondeu sem informar ao usuário que havia outras possibilidades. |
| 1    | Respondeu, mas declarou explicitamente qual interpretação adotou.                               |
| 2    | Pediu esclarecimento ao usuário ou apresentou as diferentes interpretações possíveis.           |

<BR>

| Caso | Nota | Justificativa |
| ---- | ---- | ------------- |
| T12 — "melhor município para estudar" | **0** | O sistema interpretou, sem declarar, que "melhor" significava a maior `media_geral`. Além disso, não aplicou nenhum filtro de robustez e retornou **Uru (SP), com média 741 e apenas 1 participante**. O critério adotado e a fragilidade desse resultado não foram apresentados ao usuário. Esse é exatamente o tipo de situação descrito na seção 18 (item 4.2): a resposta pode estar tecnicamente correta, mas ainda assim ser pouco útil para uma análise, especialmente quando é apresentada com a mesma confiança de um resultado mais robusto. Com um filtro de `n_participantes >= 100`, por exemplo, o resultado seria **Viçosa (MG), com média 646,74 e 535 participantes**. |
| T13 — "média de São Paulo"            | **0** | O sistema escolheu, sem explicitar, o **município de São Paulo** e a **média geral**, retornando 572,26. No entanto, "São Paulo" também pode se referir ao estado, cuja média entre os municípios é 527,89, além de existirem outras cinco médias possíveis na base. A frase "A média geral de São Paulo é 572,26" não deixa claro qual dessas interpretações foi utilizada.                                                                                                                                                                                                                                                                                                            |

Execução realizada em **07/09/2026, às 12:38:58**, utilizando `openai/gpt-oss-20b`, temperatura 0 e prompt v1.



# 18. Análise crítica do baseline

### 1. Quais requisitos o baseline atende?

De forma geral, o baseline consegue realizar as consultas tabulares para as quais foi projetado. Nos **11 casos avaliados automaticamente, todos foram aprovados**, e não houve erros de execução ou de parsing. As três perguntas que não podiam ser respondidas com os dados disponíveis também foram corretamente tratadas como abstenções.

Entre os requisitos atendidos, estão:

* **RF-01 — Correção numérica:** os resultados produzidos pelo código coincidiram com os valores de referência nos casos automáticos.
* **RF-02 — Cobertura de listas:** nos casos em que a pergunta pedia uma lista, os itens esperados foram apresentados. T03 e T04/T08, quando aplicável, tiveram cobertura de 100%.
* **RF-03 — Correção da resposta:** os valores apresentados ao usuário correspondem aos resultados da execução.
* **RF-04 — Fidelidade textual:** não foram observadas diferenças entre o resultado calculado e o valor apresentado na resposta.
* **RF-05 — Abstenção válida:** T09, T10 e T11 foram corretamente identificados como perguntas que não poderiam ser respondidas com os dados disponíveis, sem execução de código.
* **RF-06 — Evidência para auditoria:** os registros mantêm a pergunta, o código gerado, o resultado bruto e a resposta final.
* **RNF-03 — Rastreabilidade:** as principais informações da execução ficam registradas.
* **RNF-04 e RNF-05 — Estrutura e controle da resposta:** as respostas seguiram o formato previsto e os casos inviáveis foram tratados sem tentar executar código.
* **RNF-07 — Uso do dataset preparado:** as consultas são feitas sobre o dataset agregado, sem precisar processar os microdados do ENEM a cada pergunta.

Também houve um resultado positivo em relação à execução: **as 13 consultas geraram uma chamada ao LLM cada**, sem necessidade de novas tentativas.

### 2. Quais ainda não atende?

O baseline ainda não cobre alguns dos requisitos previstos para a solução completa.

O primeiro ponto é o tratamento de **ressalvas e critérios implícitos**. O sistema consegue seguir um filtro quando ele aparece claramente na pergunta, como aconteceu em T03, mas não verifica se seria necessário aplicar algum critério adicional. Isso fica evidente em T12, em que o sistema escolheu o município com maior média sem considerar o número de participantes.

Outro ponto é a **ambiguidade**. Em T12 e T13, o sistema escolheu uma interpretação possível e seguiu com ela, em vez de pedir esclarecimento ao usuário.

O requisito de **visualização (RF-08)** também ainda não é atendido, já que o baseline trabalha apenas com respostas textuais e resultados tabulares.

A **latência** também merece uma ressalva. A mediana foi de **5,16 s**, portanto abaixo do limite de 10 s, mas houve consultas com mais de 10 s. O maior tempo registrado foi de **15,84 s**.

Por fim, a execução do código ainda não está realmente isolada. Existe uma camada de validação sintática, mas ela não funciona como um sandbox completo. O código continua sendo executado no mesmo processo e não há, neste baseline, controle independente de tempo ou memória.

### 3. Quais erros ou limitações foram observados?

Não houve **erros de execução nem erros de parsing** nos 13 casos. O que apareceu foram principalmente limitações relacionadas à interpretação das perguntas.

O melhor exemplo é T12:

> “Qual é o melhor município para estudar?”

O sistema respondeu **Uru/SP, com média geral de 741**. O código gerado está correto para a interpretação escolhida — selecionar o município com maior `media_geral` —, mas a pergunta não define o que significa “melhor”. Além disso, Uru tem apenas **1 participante** no dataset.

Ou seja, o problema não está no pandas. O cálculo foi feito corretamente. O problema é que o sistema não percebeu que precisava discutir ou esclarecer o critério antes de responder.

Algo parecido aconteceu em T13:

> “Qual é a média de São Paulo?”

O sistema interpretou “São Paulo” como o município e retornou **572,26**. Essa interpretação é plausível, mas a pergunta também poderia estar se referindo ao estado. Novamente, o problema está na interpretação, não na execução do código.

Esses dois casos mostram uma limitação importante do baseline: **uma expressão pandas pode estar correta e, ainda assim, a resposta não ser necessariamente a melhor resposta para a pergunta feita**.

### 4. Quais entradas foram mais difíceis?

As consultas mais simples, como calcular uma média, contar municípios ou aplicar um filtro, foram resolvidas sem problemas. Os casos que exigiam várias operações ou alguma interpretação adicional foram mais interessantes para avaliar o comportamento do sistema.

Entre eles estão:

* **T06**, que calcula a correlação entre PIB per capita e média geral, considerando apenas municípios com pelo menos 50 participantes;
* **T07**, que primeiro seleciona os 10 municípios mais populosos e depois encontra aquele com maior média em matemática;
* **T08**, que compara as médias de duas regiões;
* **T09–T11**, que exigem reconhecer que a informação solicitada não está disponível;
* **T12–T13**, que exigem interpretar uma pergunta ambígua.

T07 é um caso particularmente útil porque a ordem das operações importa. O sistema primeiro selecionou os 10 municípios mais populosos e, dentro desse grupo, encontrou **Belo Horizonte** como o município com maior média em matemática.

Já T12 e T13 foram os casos mais difíceis do ponto de vista de interpretação, justamente porque não existe uma única resposta definida pela pergunta.

### 5. Houve resultados inesperados?

Sim. O principal foi T12.

O sistema retornou a maior média geral disponível, **741**, mas esse valor pertence a um município com apenas **1 participante**. Portanto, o resultado é válido para o cálculo realizado, mas não é uma boa resposta para uma pergunta como “qual é o melhor município para estudar?” sem alguma discussão sobre o tamanho da amostra.

Esse caso foi importante porque mostrou uma diferença entre **correção matemática** e **qualidade analítica**. O baseline sabe calcular a média e encontrar o maior valor, mas ainda não avalia se esse resultado faz sentido para a pergunta.

Outro ponto que chamou atenção foi o resultado de **11/11 nos casos automáticos**. É um resultado positivo, mas o conjunto ainda é pequeno para dizer que a arquitetura é robusta. Com apenas 11 casos, cada erro representa aproximadamente 9 pontos percentuais na taxa de aprovação.

A latência também variou bastante: ficou entre **0,65 s e 15,84 s**, com mediana de **5,16 s**. Portanto, o comportamento típico está dentro do limite de 10 s, mas alguns casos ainda são mais lentos do que o desejado.

### 6. Quais limitações decorrem do **modelo** e quais decorrem da **arquitetura**?

Nem todas as limitações observadas podem ser atribuídas ao modelo. Algumas são consequência direta de como o sistema foi construído.

**Limitações principalmente relacionadas ao modelo:**

* interpretar de forma inadequada perguntas ambíguas;
* escolher sozinho um critério para termos subjetivos, como “melhor”;
* eventualmente gerar uma expressão pandas que não corresponde exatamente à intenção do usuário;
* eventualmente ignorar uma condição da pergunta ou aplicar operações na ordem errada.

**Limitações principalmente relacionadas à arquitetura:**

* o código gerado não passa por um agente independente de validação;
* não existe mecanismo de retry para corrigir uma resposta inadequada;
* não há uma etapa específica para detectar ambiguidades;
* a execução ocorre no mesmo processo da aplicação;
* não existe isolamento de memória e tempo de execução;
* não há uma etapa dedicada à geração de visualizações;
* o texto da resposta é definido antes de o resultado da execução estar disponível.

Existe ainda uma interação entre os dois. Como o LLM é responsável por gerar o código e esse código é executado diretamente pelo sistema, um erro de interpretação do modelo pode chegar até o usuário sem que outra etapa do fluxo consiga detectá-lo.

É justamente aí que os próximos entregáveis podem mostrar um ganho importante: se uma arquitetura com Validador ou Clarificador corrigir esses casos, será possível observar o quanto a melhora veio da **estrutura do sistema**, e não apenas de uma mudança no modelo.

### 7. Alguma limitação decorre da forma como vocês **mediram** (e não do sistema)?

Sim. A principal limitação é o tamanho do conjunto de avaliação.

Os **11 casos automáticos** são suficientes para mostrar que o fluxo funciona, mas ainda são poucos para comparar arquiteturas de forma mais robusta. Uma única falha já faria a taxa cair de 100% para aproximadamente 91%. Por isso, o conjunto deverá ser ampliado no Entregável 4, com algo em torno de 25–30 casos e uma variedade maior de situações.

Outra limitação é que a avaliação verifica principalmente **o resultado final**. Isso significa que duas expressões diferentes podem, em alguns casos, produzir o mesmo resultado. Uma expressão que ignore uma restrição da pergunta poderia até passar na avaliação se, por coincidência, chegasse ao mesmo valor esperado.

Por isso, no Entregável 2, faz sentido acrescentar uma verificação estrutural do `codigo_pandas`, além da comparação do resultado.

Também é importante considerar a reprodutibilidade. A temperatura foi definida como 0, mas isso não garante que um serviço de inferência distribuído produza exatamente a mesma saída em todas as execuções. Repetir cada caso algumas vezes permitirá verificar melhor essa estabilidade.

A latência também precisa ser interpretada com cuidado. O tempo registrado representa o tempo observado para a chamada como um todo e pode incluir fatores externos ao próprio modelo, como o tempo de resposta do provedor. Assim, uma medição isolada não permite afirmar que toda a diferença entre os casos vem da complexidade da consulta.

Por fim, existe um possível viés na construção da avaliação, já que o mesmo projeto define o sistema, o prompt e os casos de teste. Esse risco é parcialmente reduzido pelo fato de os casos terem sido definidos antes da execução do baseline e de alguns testes terem sido escolhidos justamente para explorar situações menos triviais. Ainda assim, um conjunto maior e mais diversificado ajudará a reduzir esse problema.


# 19. Possíveis evoluções arquiteturais

As alternativas abaixo foram avaliadas a partir das limitações observadas na seção 18. A ideia não é adicionar componentes apenas porque são mais sofisticados, mas verificar quais deles realmente ajudam a resolver os problemas encontrados no baseline. A ordem também considera o planejamento dos próximos entregáveis.

### Adotar: Workflow (grafo explícito, LangGraph) — Entregável 2

**Resolve:** principalmente a falta de controle entre as etapas e o fato de que uma resposta pode chegar ao usuário sem uma verificação adequada.

**Como:** em vez de deixar todo o processo concentrado em uma única sequência, o sistema pode ser dividido em nós com responsabilidades claras, por exemplo: Planejador → Loader → Analisador → Validador → Sintetizador. O estado da execução fica compartilhado entre essas etapas, o que facilita acompanhar o que aconteceu e permite adicionar mecanismos de *retry* quando uma etapa falhar.

Essa estrutura também resolve uma limitação importante observada no baseline: a resposta final pode ser gerada somente depois que o resultado da consulta estiver disponível.

**Custo:** aumenta um pouco a complexidade da aplicação, principalmente pela necessidade de manter o estado do workflow e adicionar uma dependência como o LangGraph. Por outro lado, enquanto o fluxo não introduzir novas chamadas ao LLM, o impacto na latência tende a ser pequeno.

### Adotar: Validador com retry — Entregável 2

**Resolve:** a ausência de uma etapa independente que confira se o código gerado realmente responde à pergunta.

**Como:** depois que o código for gerado e executado, o sistema pode verificar algumas condições básicas, como o tipo do retorno, a faixa de valores e a ocorrência de erros. Também pode verificar aspectos mais importantes para a qualidade da consulta, como se as colunas utilizadas realmente existem no esquema e se os filtros pedidos na pergunta aparecem no código gerado.

Se a validação falhar, o resultado volta para o Analisador junto com o motivo da reprovação, permitindo que o código seja corrigido antes de produzir a resposta final.

Esse mecanismo é especialmente importante para casos como T12. O código pode estar correto do ponto de vista sintático e ainda assim não representar uma boa interpretação da pergunta. A validação não resolve toda ambiguidade possível, mas cria uma segunda oportunidade para detectar problemas antes de entregar o resultado.

**Custo:** nos casos que falharem, será necessário fazer mais uma ou duas chamadas ao LLM, aumentando a latência e o consumo de tokens. Esse custo, porém, é justamente algo que poderá ser medido no Entregável 4 para verificar se o ganho de qualidade compensa o aumento de processamento.

### Adotar: Sandbox real (subprocesso com timeout) — Entregável 2

**Resolve:** a limitação de executar o código no mesmo processo da aplicação e os problemas de segurança que não são resolvidos apenas pela validação sintática.

No baseline, a análise do código impede alguns comandos perigosos, mas não consegue controlar todos os efeitos colaterais. Um exemplo observado foi a possibilidade de utilizar `df.to_csv(...)`: `df` é um nome permitido e `to_csv` é um método legítimo do pandas, mas a operação pode produzir um efeito externo.

Além disso, executar o código no mesmo processo significa que um código com comportamento inadequado, como um laço infinito, pode afetar a própria aplicação.

**Como:** executar cada consulta em um subprocesso separado, com *timeout*, sem acesso à rede e com um diretório de trabalho restrito. A execução também pode ter stdout, stderr e exceções capturados separadamente.

Nesse caso, o isolamento não depende apenas de tentar prever todos os comandos que o LLM poderia gerar. Mesmo que apareça uma construção que não tenha sido prevista pela validação sintática, ela continua limitada pelo ambiente de execução.

**Custo:** existe um pequeno overhead para criar o processo, mas ele tende a ser baixo em relação ao tempo total de uma consulta. Em troca, o ganho em segurança e controle da execução é significativo.

### Adotar: Ferramentas (agente Visualizador) — Entregável 3

**Resolve:** principalmente o RF-08, que ainda não é atendido pelo baseline.

**Como:** depois que o resultado da consulta for validado, ele pode ser encaminhado para uma ferramenta de visualização adequada ao tipo de resultado. Por exemplo, rankings podem ser apresentados em gráficos de barras, relações entre duas variáveis podem usar gráficos de dispersão e séries podem ser representadas ao longo do tempo.

A ideia é que a visualização seja gerada a partir do resultado já validado, e não diretamente a partir da interpretação inicial do modelo.

**Custo:** será necessário adicionar ferramentas e dependências de visualização, como o `matplotlib`, além de uma etapa para decidir qual tipo de gráfico é adequado para cada resultado. O custo de processamento tende a ser pequeno, mas será necessário testar se as visualizações realmente ajudam o usuário e não apenas acrescentam complexidade.

### Adotar: Planejamento explícito — Entregável 3

**Resolve:** principalmente a falta de tratamento explícito para ambiguidades e as consultas que exigem mais de uma operação.

T07 é um bom exemplo: primeiro é necessário selecionar os 10 municípios mais populosos e, somente depois, procurar entre eles aquele com maior média em matemática. Se essas etapas forem misturadas, existe o risco de gerar uma consulta diferente da que foi solicitada.

**Como:** adicionar uma etapa antes da geração do código para classificar a pergunta como respondível, ambígua ou fora do escopo. Quando a pergunta exigir várias operações, essa etapa também pode decompor o problema em passos antes de gerar o código.

Isso também cria um ponto específico para tratar casos como T12 e T13. Em vez de simplesmente escolher uma interpretação, o sistema pode reconhecer que existe mais de uma possibilidade e solicitar esclarecimento.

**Custo:** a principal desvantagem é que o planejamento pode exigir uma chamada adicional ao LLM em todas as consultas, inclusive nas mais simples. Isso pode aumentar a latência e o consumo de tokens mesmo quando o planejamento não acrescenta nenhum benefício.

Por isso, essa é uma das mudanças que mais vale a pena medir. O ganho em casos complexos ou ambíguos precisa compensar o custo adicional nas consultas simples.

### Adotar com ressalva: Múltiplos agentes especializados

Os múltiplos agentes podem ser vistos como uma consequência natural de algumas das etapas anteriores: Planejador, Loader, Analisador, Validador, Visualizador e Sintetizador poderiam ter responsabilidades separadas.

A principal vantagem seria deixar cada parte do sistema mais bem definida. Um agente Validador, por exemplo, teria como objetivo questionar o código produzido pelo Analisador, em vez de simplesmente aceitar sua saída.

Porém, **ter vários agentes não é, por si só, uma garantia de melhoria**. Se o Sintetizador apenas reescrever uma informação que já foi produzida corretamente pelo Analisador, ele acrescenta uma chamada ao LLM sem necessariamente melhorar o resultado.

Por isso, essa alternativa deve ser adotada com cautela. No Entregável 4, seria interessante avaliar o impacto de cada componente separadamente, inclusive executando o conjunto de testes com determinado agente desligado para verificar se ele realmente contribui para a qualidade final.

### Descartar nesta fase: ReAct

O ciclo de raciocínio, ação e observação faz mais sentido quando o número de etapas necessárias para chegar à resposta não é conhecido antecipadamente.

Neste projeto, o fluxo principal já é relativamente previsível: interpretar a pergunta → consultar os dados → validar o resultado → gerar a resposta. Um workflow explícito consegue representar esse processo de forma mais controlada.

O ReAct também poderia gerar um número variável de chamadas ao LLM, o que dificulta prever a latência e o custo. Para o problema atual, isso parece adicionar complexidade sem resolver diretamente as principais limitações encontradas.

Por isso, **ReAct é descartado nesta fase**. Ele poderia voltar a ser considerado caso o sistema passe a trabalhar com fontes externas ou consultas nas quais o número de etapas necessárias seja realmente imprevisível.

### Descartar nesta fase: Memória de sessão

A memória seria útil para perguntas de acompanhamento, como “e no ano anterior?” ou “e considerando apenas os municípios com mais de 50 participantes?”. Nesse caso, informações de perguntas anteriores precisariam ser mantidas para interpretar corretamente a próxima consulta.

Esse cenário, porém, **não está no escopo atual nem aparece no conjunto de avaliação**. Adicionar memória agora tornaria os testes mais difíceis de controlar, já que cada caso poderia depender do contexto das perguntas anteriores, sem necessariamente melhorar nenhum dos resultados que estão sendo medidos.

Por isso, a memória fica **adiada para uma etapa posterior**, caso o sistema passe a trabalhar com conversas de múltiplas perguntas e a taxa de acerto das consultas independentes já esteja estável.

### Descartar nesta fase: MCP

Um servidor MCP poderia ser interessante se o sistema precisasse integrar diferentes fontes de dados e ferramentas. Por exemplo, poderia facilitar a comunicação entre o modelo e uma API externa, ferramentas de consulta ou serviços de visualização.

No entanto, essa necessidade ainda não apareceu na avaliação do baseline. O dataset atual já está preparado e foi suficiente para responder aos casos avaliados. Portanto, introduzir MCP neste momento adicionaria uma camada de infraestrutura sem resolver nenhuma limitação observada diretamente.

Assim, **MCP é descartado para os Entregáveis 2 e 3**. Pode ser reavaliado no Entregável 4 caso o escopo de dados seja ampliado ou o sistema passe a trabalhar com diferentes fontes.

### Descartar nesta fase: RAG / busca semântica

O uso de RAG ou busca semântica também não parece necessário para este problema. O sistema trabalha com **dados tabulares**, e as informações relevantes para interpretar as consultas e o esquema do dataset já cabem no contexto utilizado pelo modelo.

Não existe, portanto, um corpus textual grande que precise ser recuperado dinamicamente para responder às perguntas avaliadas.

Por isso, **RAG é descartado nesta fase**.


# 20. Pergunta obrigatória

> **Como o grupo pretende demonstrar, ao final do curso, que a arquitetura final apresenta
> vantagens em relação ao baseline?**

**Hipótese:** no mesmo conjunto de perguntas, a arquitetura final deverá reduzir principalmente os **erros silenciosos** — respostas incorretas apresentadas como válidas — e melhorar o tratamento de ambiguidades e consultas mais complexas, mantendo o aumento de latência e de custo dentro de limites aceitáveis.

Como o baseline apresentou **11/11 de aprovação nos casos automáticos**, a taxa de acerto isoladamente não será suficiente para demonstrar a melhoria. A principal evidência será a capacidade da arquitetura final de identificar situações como T12 e T13, nas quais o código executado é válido, mas a interpretação da pergunta é inadequada ou ambígua.

A comparação será feita com o **mesmo conjunto de testes**, ampliado para 25–30 casos no Entregável 4, com o baseline reexecutado sobre esse conjunto. Serão consideradas as seguintes métricas:

* **taxa de aprovação**, geral e por tipo de caso;
* **taxa de erro silencioso**;
* **taxa de abstenção correta** e de falsas abstenções;
* **tratamento de ambiguidades**, especialmente a capacidade de declarar a interpretação ou pedir esclarecimento;
* **fidelidade entre o resultado executado e a resposta apresentada**;
* **latência mediana** e **número de chamadas/tokens ao LLM**.

Além disso, será realizada uma **ablação dos componentes da arquitetura**, permitindo verificar se cada agente ou etapa realmente contribui para o resultado final.

A arquitetura será considerada vantajosa se apresentar **melhoria de qualidade e redução de erros silenciosos**, sem que o ganho implique um aumento desproporcional de latência ou custo. Caso isso não ocorra, a conclusão será de que a complexidade adicional da arquitetura não trouxe benefício suficiente para o problema avaliado.



# 21. Conclusão

**Problema.** Os dados públicos brasileiros de educação, como os do ENEM/INEP, e os dados de contexto municipal, como os do IBGE, são abertos, mas seu uso ainda exige algum conhecimento técnico. O objetivo do sistema é justamente reduzir essa barreira, permitindo que um analista de secretaria, jornalista de dados ou gestor escolar faça perguntas em português e obtenha respostas a partir desses dados sem precisar escrever consultas em pandas.

**Preparação dos dados.** Para isso, foi desenvolvido um ETL executado uma única vez, fora do sistema de consulta. Ele reduziu os 1,8 GB de microdados a um artefato de aproximadamente 0,6 MB, contendo 5.481 municípios e 721.429 participantes, já cruzados com dados de população e PIB do IBGE. Essa foi uma decisão de arquitetura importante, pois faz com que o custo de cada consulta deixe de depender do tamanho da base bruta. O cruzamento entre INEP e IBGE, que havia sido apontado como um risco na proposta, funcionou como esperado, com correspondência 1:1 para todos os municípios. O principal problema encontrado acabou sendo outro: definir corretamente quem deve ser considerado participante.

**Baseline.** O baseline foi construído com uma única chamada ao LLM. O modelo produz um plano estruturado, indicando se a pergunta pode ser respondida, qual expressão pandas deve ser utilizada e qual resposta deve ser apresentada. A execução e a formatação são então feitas de forma determinística. O resultado é um baseline **parcial**: ele atende às consultas numéricas, rankings, cruzamentos ENEM × IBGE e abstenções previstas nos requisitos RF-01 a RF-06, mas ainda não contempla visualizações (RF-08), ressalvas dependentes da consulta (RF-07) nem tratamento explícito de ambiguidades (RF-09).

**Resultados.** Na execução realizada em 07/09/2026, utilizando `openai/gpt-oss-20b`, temperatura 0 e o prompt v1, os 11 casos automáticos foram aprovados. Isso inclui os três cruzamentos ENEM × IBGE e as três perguntas que deveriam resultar em abstenção. Não houve erros de execução ou de parsing, e foram feitas 13 chamadas ao modelo, uma para cada caso. Ao todo, foram registrados 17.295 tokens de entrada e 7.597 de saída, com custo estimado de US$ 0,0055.

Apesar do bom resultado nos casos automáticos, a latência mediana foi de **11,51 s**, acima da meta de 10 s. A distribuição bimodal observada sugere que parte desse tempo pode estar relacionada à fila do provedor gratuito, e não apenas à complexidade das consultas.

Os casos manuais mostraram uma limitação mais importante. Tanto T12 quanto T13 receberam nota **0**. Em T12, a pergunta “Qual é o melhor município para estudar?” levou o sistema a responder **Uru (SP), com média 741 e apenas 1 participante**. O cálculo está correto para a operação realizada, mas a resposta não considera que “melhor” exige algum critério adicional. Em T13, o sistema interpretou “São Paulo” como município e apresentou sua média sem deixar explícita essa interpretação. Esses casos mostram que **acertar o cálculo não significa necessariamente responder corretamente à pergunta**.

Também houve um problema inicial no modo de *tool calling*: a primeira tentativa de execução retornou HTTP 400. O problema foi resolvido alterando o método para `json_schema`, sem necessidade de modificar o restante do fluxo.

**Limitações.** A avaliação permitiu separar melhor as limitações do **modelo** das limitações da **arquitetura**. No modelo, estão principalmente os erros na tradução da pergunta para pandas, como ignorar filtros, escolher uma coluna inadequada ou aplicar operações na ordem errada. Na arquitetura, estão a ausência de uma etapa independente de validação, a geração da frase antes de o resultado estar disponível, o tratamento fixo de ressalvas, a falta de detecção de ambiguidades e a execução do código no mesmo processo da aplicação.

Essa distinção é importante porque as próximas etapas do projeto podem atacar principalmente as limitações arquiteturais. O objetivo não é apenas trocar o modelo e observar se a taxa de acerto muda, mas verificar se uma arquitetura mais controlada consegue impedir que erros de interpretação cheguem ao usuário.

A própria avaliação também tem limitações. Os 11 casos automáticos ainda são poucos, a comparação do resultado final não garante que o caminho utilizado pelo modelo estava correto e uma única execução por caso não permite separar completamente uma melhoria real de uma variação ocasional do modelo.

**Próxima etapa.** A principal hipótese para a próxima versão é que um **Validador com retry**, no Entregável 2, trará o maior ganho em relação ao custo. Ele ataca diretamente uma das principais limitações encontradas: permitir que uma resposta inadequada, como a de T12, chegue ao usuário sem nenhuma segunda verificação. Como o custo adicional ocorre principalmente quando uma execução é reprovada, o impacto sobre as consultas que já funcionam deve ser pequeno.

Como o baseline já atingiu 11/11 nos casos automáticos, a melhoria da arquitetura final não poderá ser demonstrada apenas por uma taxa de acerto maior nesse conjunto. A comparação deverá se concentrar principalmente em **erros silenciosos, tratamento de ambiguidades e casos mais complexos**, utilizando um conjunto de avaliação ampliado.

O **Planejador explícito**, previsto para o Entregável 3, também será avaliado, mas apresenta uma relação custo-benefício menos clara: ele adiciona uma chamada ao LLM em todas as consultas, inclusive nas mais simples. Por isso, mais do que assumir que uma arquitetura mais complexa será melhor, o objetivo dos próximos entregáveis é **medir se cada componente realmente melhora o sistema e se esse ganho compensa o custo adicional**.


---

# Checklist antes da entrega

- [x] O problema está claramente definido. *(seção 1)*
- [x] O usuário-alvo foi identificado. *(seção 2)*
- [x] Escopo e não-objetivos estão explícitos. *(seção 4)*
- [x] Existem requisitos funcionais **verificáveis**. *(seção 6, com a coluna "como será verificado")*
- [x] Existem requisitos não funcionais. *(seção 7, com valores de referência)*
- [x] Cada critério de sucesso diz **como** será medido. *(seção 10)*
- [x] O tipo de baseline foi classificado e justificado. *(seção 9 — parcial)*
- [x] O baseline executa sem erros. *(21 células executadas, 0 erros — 07/09/2026)*
- [x] Existem pelo menos três casos de teste, cobrindo mais de um tipo. *(13 casos, 7 tipos)*
- [x] Modelo, temperatura e data da execução estão registrados. *(`RUN_INFO`, seção 11)*
- [x] Latência, tokens e número de chamadas foram registrados. *(seção 16)*
- [x] Os resultados estão na tabela e interpretados. *(seções 17 e 18)*
- [x] As limitações foram analisadas. *(seção 18, separadas por origem)*
- [x] A pergunta obrigatória foi respondida. *(seção 20)*
- [x] O notebook foi executado do início ao fim e **salvo com as saídas**. *(07/09/2026 12:38)*
- [x] O notebook pode ser executado por outra pessoa. *(seção 12 baixa o artefato; código do ETL na mesma seção)*
- [x] Nenhuma chave de API foi incluída no notebook. *(seção 11)*
